# Paper combined - train the edit model, then run BCO experiments

**PART 1** fine-tunes the final-experiments RTT + connectivity agent for 100 epochs on the current 500-graph curriculum with an adjustment-degree cap penalty (W=10, target=0.2, paper mode, demand off). **PART 2** runs the paper experiments (E1-E6) using that fine-tuned model as `OUR_MODEL_PATH`. Algorithm budgets are set for a full top-to-bottom run that should fit into roughly 25-30 hours on the current CPU calibration; rerun the timing estimator cell after any budget change.

Run top-to-bottom: training must finish before the experiment config picks up the trained weights.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # select GPU 1
import sys, pickle, shutil, json, random as _random
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from hydra import compose, initialize_config_dir
from tqdm.auto import tqdm
from IPython.display import Image, display

from eval_lib.context import (ROOT_DIR, CFG_DIR, DATASETS_DIR,
                              MODEL_OUTPUTS_DIR, EDIT_MODEL_WEIGHTS_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from connectpt.routes_generator import utils as lrnu
from connectpt.routes_generator.improvement_learning import (
    _get_planned_current_routes, _make_route_context_state,
    load_raw_graphs_and_lc_routes, make_improvement_batch,
    rollout_lc_improvement, train_lc_improvement_cfg)
from connectpt.routes_generator.torch_utils import (
    get_batch_tensor_from_routes, dump_routes)
from connectpt.routes_generator.transit_time_estimator import (
    ROUTE_ACTION_EXTEND, ROUTE_ACTION_HALT, ROUTE_ACTION_TRIM_END,
    ROUTE_ACTION_TRIM_START, RouteGenBatchState)
from connectpt.routes_generator.citygraph_dataset import (
    STOP_KEY, DynamicCityGraphDataset)
from connectpt.routes_generator.bee_colony import get_adjustment_degrees
from torch_geometric.data import Batch
from eval_lib.results_io import save_table
from eval_lib import build_lc_cfg, run_lc, as_route_tensor
from eval_lib import plots as route_plots

pd.set_option("display.max_columns", None)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [2]:
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

from eval_lib.context import ROOT_DIR, EDIT_MODEL_WEIGHTS_DIR
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from eval_lib import *                      # runners, builders, BENCHMARK_SPECS, ...
from eval_lib import _run_baseline          # private: not pulled by `import *`
from eval_lib import plots as route_plots
from connectpt.routes_generator.bee_colony import get_adjustment_degrees

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pd.set_option("display.max_columns", None)
print("device:", device)

# --- wall-clock instrumentation (per-epoch / per-iteration) ---
import time as _time
TIMING = {"epoch_s": None, "dataset_graph_s": None, "bco_iter_s": {}, "base_iter_s": {}}


device: cpu


## Configuration

`copy_full`, `copy_boundary`, `copy_mixed`, and `lc_clean` are equal-size
tiers.  The three corrupted tiers are generated from LC routes with the four
copy/subcopy mutations below.  Multiplicity grows across the curriculum up to
five routes on one stop-to-stop leg.

In [3]:
# Training is OFF: experiments use an existing checkpoint (see OUR_MODEL_PATH).
# Set True to (re)train the edit model in PART 1.
RUN_TRAINING = False

# --- dataset: five tiers (four corrupted + clean), easy -> hard/clean ---
N_GRAPHS        = 500   # full training pool: 100 graphs/tier
RAW_N_NODES     = 50
RAW_GRAPH_TYPE  = "mixed"
RAW_GRAPH_SEED  = 0
TARGET_N_ROUTES = 12
MIN_ROUTE_LEN   = 8
MAX_ROUTE_LEN   = 15
LC_N_SAMPLES    = 1
LC_COMBOS = [(1.0,0.0,0.0,"demand"), (0.0,1.0,0.0,"route"), (0.0,0.0,1.0,"conn")]

# Cap on unserved demand (%) for the coverage-preserving duplicate tier: a copy
# is only injected if the network still leaves <= this much demand disconnected.
# Forces the agent to dedup (cut RTT) instead of extend-to-cover.
D_UN_TARGET_PCT = 10.0

COPY_MUTATION_KINDS = ("full_copy", "prefix_copy", "suffix_copy", "middle_copy")
TIERS = ["copy_full", "copy_boundary", "copy_mixed", "covered_dup", "lc_clean"]
TIER_CFG = {
    "copy_full": {"events": 3, "max_multiplicity": 3, "kinds": ("full_copy",)},
    "copy_boundary": {"events": 4, "max_multiplicity": 4, "kinds": ("prefix_copy", "suffix_copy")},
    "copy_mixed": {"events": 6, "max_multiplicity": 5, "kinds": COPY_MUTATION_KINDS},
    "covered_dup": {
        # duplicates injected ONLY where coverage stays intact (d_un <= cap),
        # so trimming them is the only way to cut cost.
        "events": 6, "max_multiplicity": 4, "kinds": COPY_MUTATION_KINDS,
        "covered": True, "d_un_cap": D_UN_TARGET_PCT,
    },
    "lc_clean": {"events": 0, "max_multiplicity": 1, "kinds": ()},
}
DATASET_DIRNAME = "lc_copy_subcopy_curriculum_n500_n50_r12_len8_15_v4"
NEW_DATASET_DIR = DATASETS_DIR / DATASET_DIRNAME
SUBSET_PKL = NEW_DATASET_DIR / "raw_graphs_subset.pkl"
META_CSV   = NEW_DATASET_DIR / "meta.csv"
FORCE_REGEN = False

# --- objective route + connectivity + adj fine-tuning ---
DISABLED_COST_COMPONENTS = ["demand"]
CONNECTIVITY_MODE = "weighted_median"
ADJ_MODE = "paper"; ADJ_GAP = 0.1
VARY_WEIGHTS = False; OP_FRACTION = 0.4; MCW_FRACTION = 0.4
ROUTE_W = 0.5; CONN_W = 0.5            # fixed RTT + connectivity objective (0.5/0.5)
ADJ_TRAIN_WEIGHT = 10.0; ADJ_TRAIN_TARGET = 0.2; ADJ_TRAIN_OBJECTIVE = "cap"
# Adj semantics are intentionally identical in training and BCO sweeps: paper-mode cap penalty.

# --- anti-halt-collapse ---
FORCE_NONHALT_FIRST_STEP = True
FORCE_NONHALT_UNTIL_ITER = 12
POSITIVE_ONLY_TRIM_REWARD = False
ZERO_TRIM_REWARD = False
ENTROPY_WEIGHT = 0.01

# --- critic norm + Huber + value clip (both runs) ---
CRITIC_OVERRIDES = ["++critic_normalize_returns=true", "++critic_huber=true",
                    "++critic_huber_delta=1.0", "++critic_value_clip=0.2"]

# --- adj fine-tune run ---
N_ITERATIONS = 100   # adj fine-tuning epochs from the final rtt+conn checkpoint
BATCH_SIZE   = 4     # CPU-budgeted; raise only after re-running the estimator
TRAIN_FRACTION = 0.9
SPLIT_SEED   = 0
MAX_ROUTE_EDIT_STEPS = MAX_ROUTE_LEN
MAX_TRIM_ACTIONS_PER_ROUTE = 1

# --- initialize from the parent PRESERVED rtt+wmc checkpoint; optionally resume this fine-tune output ---
RESUME_FROM_CHECKPOINT = False
RESUME_TAG = "resume"
# Fine-tune base = parent PRESERVED rtt+wmc checkpoint (no fallback list).
BASE_MODEL_PATH = EDIT_MODEL_WEIGHTS_DIR / "improvement_lc_redundancy_rttwmc_v1_PRESERVED.pt"
REQUIRE_BASE_MODEL = True


# --- cumulative curriculum: duplicate -> partial -> mixed -> covered -> clean ---
USE_CURRICULUM = True
CURRICULUM = [
    (round(0.12 * N_ITERATIONS), ["copy_full"],                                              "full-copy"),
    (round(0.22 * N_ITERATIONS), ["copy_full", "copy_boundary"],                             "+boundary"),
    (round(0.36 * N_ITERATIONS), ["copy_full", "copy_boundary", "copy_mixed"],               "+mixed"),
    (round(0.52 * N_ITERATIONS), ["copy_full", "copy_boundary", "copy_mixed", "covered_dup"],"+covered"),
    (N_ITERATIONS,               TIERS,                                                       "+clean"),  # last ~48% includes lc_clean
]

# --- evaluation ---
BALANCED_EVAL_WEIGHTS = (0.5, 0.5)
TRAIN_EVAL_N_PER_TIER = 4   # balanced validation monitor graphs per tier
EVAL_N_PER_TIER = 10
RUN_NAME = "lc_rttconn_adj_w10_t02_finetune100"
HISTORY_CHECKPOINT_PATH = MODEL_OUTPUTS_DIR / f"{RUN_NAME}_training_history_partial.csv"
FULL_HISTORY_RUN = f"{RUN_NAME}_{RESUME_TAG}" if RESUME_FROM_CHECKPOINT else RUN_NAME
FULL_HISTORY_CHECKPOINT = MODEL_OUTPUTS_DIR / f"{FULL_HISTORY_RUN}_training_history_partial.csv"
PRIOR_HISTORY_FILES = [HISTORY_CHECKPOINT_PATH] if RESUME_FROM_CHECKPOINT else []

print(f"{N_GRAPHS} graphs, tiers={TIERS}; curriculum={[c[2]+'<'+str(c[0]) for c in CURRICULUM]}")
print(f"covered_dup d_un cap = {D_UN_TARGET_PCT}%")
print(f"FINE-TUNE: {N_ITERATIONS} epochs, batch={BATCH_SIZE}, graphs={N_GRAPHS}, resume={RESUME_FROM_CHECKPOINT}")
print(f"base model -> {BASE_MODEL_PATH} | exists={BASE_MODEL_PATH.exists()}")
print(f"connectivity_mode -> {CONNECTIVITY_MODE}")
print(f"plain reward (positive_only={POSITIVE_ONLY_TRIM_REWARD}, zero_trim={ZERO_TRIM_REWARD})")
print(f"dataset -> {NEW_DATASET_DIR}")


500 graphs, tiers=['copy_full', 'copy_boundary', 'copy_mixed', 'covered_dup', 'lc_clean']; curriculum=['full-copy<12', '+boundary<22', '+mixed<36', '+covered<52', '+clean<100']
covered_dup d_un cap = 10.0%
FINE-TUNE: 100 epochs, batch=4, graphs=500, resume=False
base model -> D:\PythonProjects\connectpt\artifacts\model_weights\improvement\improvement_lc_redundancy_rttwmc_v1_PRESERVED.pt | exists=True
connectivity_mode -> weighted_median
plain reward (positive_only=False, zero_trim=False)
dataset -> D:\PythonProjects\connectpt\datasets\lc_copy_subcopy_curriculum_n500_n50_r12_len8_15_v4


## Generate the four-tier route-copy dataset

Every corruption copies a whole donor route or a contiguous donor subroute
into another route slot.  Prefix, suffix, and interior replacements preserve
the recipient length.  Candidates with repeated stops are rejected, so the
dataset teaches inter-route redundancy rather than synthetic self-loops.

In [4]:
if RUN_TRAINING:
    def _to_fixed(routes):
        t = as_route_tensor(routes).long()
        if t.ndim == 3:
            t = t[0]
        if t.shape[0] < TARGET_N_ROUTES:
            t = torch.cat([t, torch.full((TARGET_N_ROUTES - t.shape[0], t.shape[1]), -1, dtype=t.dtype)], 0)
        else:
            t = t[:TARGET_N_ROUTES]
        if t.shape[1] < MAX_ROUTE_LEN:
            t = torch.cat([t, torch.full((t.shape[0], MAX_ROUTE_LEN - t.shape[1]), -1, dtype=t.dtype)], 1)
        elif t.shape[1] > MAX_ROUTE_LEN:
            t = t[:, :MAX_ROUTE_LEN]
        return t


    def _tensors(g):
        return {"node_locs": g[STOP_KEY].pos.detach().cpu().clone(),
                "street_adj": g.street_adj.detach().cpu().clone(),
                "demand": g.demand.detach().cpu().clone()}


    def _route_nodes(route):
        return [int(node) for node in route.tolist() if int(node) >= 0]


    def _leg_counts(routes):
        counts = Counter()
        for route in routes:
            nodes = _route_nodes(route)
            for start, end in zip(nodes[:-1], nodes[1:]):
                counts[(min(start, end), max(start, end))] += 1
        return counts


    def _redundancy_stats(routes):
        counts = _leg_counts(routes)
        traversals = sum(counts.values())
        redundancy = 0.0 if traversals == 0 else (traversals - len(counts)) / traversals
        return {
            "redundancy": float(redundancy),
            "max_leg_use": max(counts.values(), default=0),
            "edge_traversals": traversals,
            "unique_edges": len(counts),
        }


    def _uncovered_demand_pct(routes, demand, n_nodes):
        '''Demand-weighted % of OD pairs not connected by any route path.'''
        parent = list(range(n_nodes))
        def find(x):
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:
                parent[x], x = root, parent[x]
            return root
        for route in routes:
            ns = _route_nodes(route)
            for a, b in zip(ns[:-1], ns[1:]):
                ra, rb = find(a), find(b)
                if ra != rb:
                    parent[ra] = rb
        D = demand if torch.is_tensor(demand) else torch.as_tensor(demand)
        idx = (D > 0).nonzero(as_tuple=False).tolist()
        total = uncovered = 0.0
        for a, b in idx:
            if a == b:
                continue
            w = float(D[a, b]); total += w
            if find(a) != find(b):
                uncovered += w
        return 0.0 if total == 0 else 100.0 * uncovered / total


    def _is_simple_route(nodes):
        return (MIN_ROUTE_LEN <= len(nodes) <= MAX_ROUTE_LEN and
                len(nodes) == len(set(nodes)) and
                all(a != b for a, b in zip(nodes[:-1], nodes[1:])))


    def _copy_candidate(routes, donor_idx, target_idx, kind, rng):
        donor = _route_nodes(routes[donor_idx])
        target = _route_nodes(routes[target_idx])
        if not donor or not target:
            return None
        if kind == "full_copy":
            candidate = donor
        else:
            max_seg_len = min(len(donor), len(target), 6)
            if kind == "middle_copy":
                max_seg_len = min(max_seg_len, len(target) - 2)
            if max_seg_len < 2:
                return None
            seg_len = rng.randint(2, max_seg_len)
            if kind == "prefix_copy":
                candidate = donor[:seg_len] + target[seg_len:]
            elif kind == "suffix_copy":
                candidate = target[:-seg_len] + donor[-seg_len:]
            elif kind == "middle_copy":
                donor_start = rng.randint(0, len(donor) - seg_len)
                target_start = rng.randint(1, len(target) - seg_len - 1)
                candidate = (target[:target_start] +
                             donor[donor_start:donor_start + seg_len] +
                             target[target_start + seg_len:])
            else:
                raise ValueError(f"unknown mutation kind: {kind}")
        if candidate == target or not _is_simple_route(candidate):
            return None
        return candidate


    def _replace_route(routes, route_idx, nodes):
        routes[route_idx] = -1
        routes[route_idx, :len(nodes)] = torch.as_tensor(nodes, dtype=routes.dtype)


    def _try_copy_mutation(routes, kind, max_multiplicity, rng, attempts=80):
        '''Copy one donor route/subroute into 1..max_multiplicity-1 recipients.'''
        base = routes.clone()
        for _ in range(attempts):
            donor_idx = rng.randrange(routes.shape[0])
            target_idxs = [idx for idx in range(routes.shape[0]) if idx != donor_idx]
            rng.shuffle(target_idxs)
            wanted = rng.randint(1, min(max_multiplicity - 1, len(target_idxs)))
            mutated = base.clone()
            current_redundancy = _redundancy_stats(mutated)["redundancy"]
            touched = 0
            for target_idx in target_idxs:
                candidate = _copy_candidate(mutated, donor_idx, target_idx, kind, rng)
                if candidate is None:
                    continue
                proposal = mutated.clone()
                _replace_route(proposal, target_idx, candidate)
                proposal_redundancy = _redundancy_stats(proposal)["redundancy"]
                if proposal_redundancy <= current_redundancy + 1e-12:
                    continue
                mutated = proposal
                current_redundancy = proposal_redundancy
                touched += 1
                if touched >= wanted:
                    return mutated, touched
        return base, 0


    def _try_covered_copy_mutation(routes, kind, max_multiplicity, rng,
                                   demand, n_nodes, d_un_cap, attempts=160):
        '''Inject a duplicate only if coverage stays intact (d_un <= cap).'''
        base = routes.clone()
        for _ in range(attempts):
            donor_idx = rng.randrange(routes.shape[0])
            target_idxs = [idx for idx in range(routes.shape[0]) if idx != donor_idx]
            rng.shuffle(target_idxs)
            wanted = rng.randint(1, min(max_multiplicity - 1, len(target_idxs)))
            mutated = base.clone()
            current_redundancy = _redundancy_stats(mutated)["redundancy"]
            touched = 0
            for target_idx in target_idxs:
                candidate = _copy_candidate(mutated, donor_idx, target_idx, kind, rng)
                if candidate is None:
                    continue
                proposal = mutated.clone()
                _replace_route(proposal, target_idx, candidate)
                if _redundancy_stats(proposal)["redundancy"] <= current_redundancy + 1e-12:
                    continue
                if _uncovered_demand_pct(proposal, demand, n_nodes) > d_un_cap:
                    continue  # would break coverage -> reject
                mutated = proposal
                current_redundancy = _redundancy_stats(mutated)["redundancy"]
                touched += 1
                if touched >= wanted:
                    return mutated, touched
        return base, 0


    def inject_route_copy_redundancy(routes, tier_cfg, rng, demand=None, n_nodes=None):
        routes = routes.clone()
        applied_events = Counter()
        mutated_routes = Counter()
        allowed_kinds = tuple(tier_cfg["kinds"])
        covered = bool(tier_cfg.get("covered", False))
        d_un_cap = float(tier_cfg.get("d_un_cap", 100.0))
        for _ in range(int(tier_cfg["events"])):
            candidates = list(allowed_kinds)
            rng.shuffle(candidates)
            for kind in candidates:
                if covered:
                    proposal, touched = _try_covered_copy_mutation(
                        routes, kind, int(tier_cfg["max_multiplicity"]), rng,
                        demand, n_nodes, d_un_cap)
                else:
                    proposal, touched = _try_copy_mutation(
                        routes, kind, int(tier_cfg["max_multiplicity"]), rng)
                if touched:
                    routes = proposal
                    applied_events[kind] += 1
                    mutated_routes[kind] += touched
                    break
        return routes, applied_events, mutated_routes


    def generate_dataset():
        if NEW_DATASET_DIR.exists():
            shutil.rmtree(NEW_DATASET_DIR)
        NEW_DATASET_DIR.mkdir(parents=True, exist_ok=True)
        _random.seed(RAW_GRAPH_SEED); torch.manual_seed(RAW_GRAPH_SEED)
        ds = DynamicCityGraphDataset(min_nodes=RAW_N_NODES, max_nodes=RAW_N_NODES,
                                     data_type=RAW_GRAPH_TYPE, mumford_style=True, pos_only=False)
        raw = [ds.generate_graph(n_nodes=RAW_N_NODES) for _ in range(N_GRAPHS)]
        per = N_GRAPHS // len(TIERS)
        subset, meta = [], []
        for gi, g in enumerate(tqdm(raw, desc="generate dataset")):
            tier = TIERS[min(gi // per, len(TIERS) - 1)]
            tier_cfg = TIER_CFG[tier]
            rng = _random.Random(1000 + gi)
            d, rt, cn, ctag = LC_COMBOS[gi % len(LC_COMBOS)]
            c = build_lc_cfg(run_name=f"copy_cur_{gi}", n_routes=TARGET_N_ROUTES,
                             min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN,
                             demand_time_weight=d, route_time_weight=rt,
                             median_connectivity_weight=cn,
                             connectivity_mode=CONNECTIVITY_MODE)
            _, _, _, raw_routes, _ = run_lc(c, tensors=(tn := _tensors(g)), run_name_prefix="copy_cur_",
                                            n_samples=LC_N_SAMPLES)
            routes = _to_fixed(raw_routes)
            before = _redundancy_stats(routes)
            routes, event_counts, route_counts = inject_route_copy_redundancy(
                routes, tier_cfg, rng, demand=tn["demand"], n_nodes=RAW_N_NODES)
            after = _redundancy_stats(routes)
            gdir = NEW_DATASET_DIR / f"graph_{gi:04d}"; gdir.mkdir(parents=True, exist_ok=True)
            dump_routes(f"lc_copy_cur_graph_{gi:04d}_routes", routes, out_dir=gdir)
            subset.append(g)
            meta.append({
                "graph_index": gi, "tier": tier, "lc_combo": ctag,
                "requested_events": int(tier_cfg["events"]),
                "applied_events": int(sum(event_counts.values())),
                "mutated_routes": int(sum(route_counts.values())),
                "mutation_events": json.dumps(dict(event_counts), sort_keys=True),
                "mutation_routes": json.dumps(dict(route_counts), sort_keys=True),
                "d_un_after_pct": round(_uncovered_demand_pct(routes, tn["demand"], RAW_N_NODES), 2),
                "redun_before": round(before["redundancy"], 4),
                "redun_after": round(after["redundancy"], 4),
                "max_leg_use_before": before["max_leg_use"],
                "max_leg_use_after": after["max_leg_use"],
            })
            if (gi + 1) % 100 == 0:
                print(f"  {gi+1}/{N_GRAPHS} (tier={tier})")
        with SUBSET_PKL.open("wb") as fh:
            pickle.dump(subset, fh)
        pd.DataFrame(meta).to_csv(META_CSV, index=False)
        print(f"Saved {len(subset)} graphs -> {NEW_DATASET_DIR}")


    _have = len(list(NEW_DATASET_DIR.glob("graph_*"))) if NEW_DATASET_DIR.exists() else 0
    if SUBSET_PKL.exists() and _have == N_GRAPHS and META_CSV.exists() and not FORCE_REGEN:
        print(f"dataset already exists ({_have}) -> skip")
    else:
        if _have and _have != N_GRAPHS:
            print(f"found {_have} graphs, expected {N_GRAPHS} -> regenerate")
        _tg = _time.perf_counter()
        generate_dataset()
        TIMING["dataset_graph_s"] = (_time.perf_counter() - _tg) / max(1, N_GRAPHS)
        print(f"[timing] dataset gen: {TIMING['dataset_graph_s']:.2f} s/graph "
              f"(scales with N_GRAPHS: {N_GRAPHS} now)")

## Load, split, and define the cumulative curriculum

In [5]:
if RUN_TRAINING:
    graphs, seed_routes = load_raw_graphs_and_lc_routes(SUBSET_PKL, NEW_DATASET_DIR)
    meta_df = pd.read_csv(META_CSV)
    N = len(graphs)
    print(f"loaded {N} graphs; seed_routes={tuple(seed_routes.shape)}")
    print("copy/subcopy corruption summary by tier:")
    display(meta_df.groupby("tier")[[
        "requested_events", "applied_events", "mutated_routes",
        "redun_before", "redun_after", "max_leg_use_after", "d_un_after_pct",
    ]].mean().round(3).reindex(TIERS))

    _perm = torch.randperm(N, generator=torch.Generator().manual_seed(SPLIT_SEED))
    TIER_OF = dict(zip(meta_df["graph_index"], meta_df["tier"]))

    # Stratified split: each tier contributes its own train/val so EVERY tier has
    # >= TRAIN_EVAL_N_PER_TIER validation graphs (robust for small N / smoke runs).
    _by_tier = {tier: [] for tier in TIERS}
    for gi in _perm.tolist():
        _by_tier[TIER_OF[gi]].append(gi)
    _train_l, _val_l = [], []
    for tier in TIERS:
        idxs = _by_tier[tier]
        n_val = max(TRAIN_EVAL_N_PER_TIER, int(round((1 - TRAIN_FRACTION) * len(idxs))))
        n_val = min(n_val, max(0, len(idxs) - 1))        # keep >= 1 train when possible
        if len(idxs) - n_val < 1 or n_val < TRAIN_EVAL_N_PER_TIER:
            raise ValueError(
                f"tier '{tier}' has only {len(idxs)} graphs -- too few for "
                f"TRAIN_EVAL_N_PER_TIER={TRAIN_EVAL_N_PER_TIER}; raise N_GRAPHS or "
                f"lower TRAIN_EVAL_N_PER_TIER")
        _val_l += idxs[:n_val]; _train_l += idxs[n_val:]
    TRAIN_INDICES = torch.tensor(_train_l, dtype=torch.long)
    VAL_INDICES = torch.tensor(_val_l, dtype=torch.long)

    _val_by_tier = {tier: [] for tier in TIERS}
    for gi in VAL_INDICES.tolist():
        _val_by_tier[TIER_OF[gi]].append(gi)
    MONITOR_VAL_INDICES = torch.tensor([
        gi for tier in TIERS for gi in _val_by_tier[tier][:TRAIN_EVAL_N_PER_TIER]
    ], dtype=torch.long)
    print(f"validation graphs={len(VAL_INDICES)}; balanced train-loop monitor={len(MONITOR_VAL_INDICES)}")

    _train_by_tier = {tier: [] for tier in TIERS}
    for gi in TRAIN_INDICES.tolist():
        _train_by_tier[TIER_OF[gi]].append(gi)
    _train_by_tier = {
        tier: torch.tensor(indices, dtype=torch.long)
        for tier, indices in _train_by_tier.items()
    }
    print("train graphs per tier:", {tier: len(indices) for tier, indices in _train_by_tier.items()})


    def curriculum_fn(iteration):
        '''iteration -> (active train indices, stage label).'''
        for until, tiers, label in CURRICULUM:
            if iteration < until:
                idx = torch.cat([_train_by_tier[tier] for tier in tiers if len(_train_by_tier[tier])])
                return idx, label
        tiers = CURRICULUM[-1][1]
        idx = torch.cat([_train_by_tier[tier] for tier in tiers if len(_train_by_tier[tier])])
        return idx, CURRICULUM[-1][2]


    def stage_spans():
        '''[(start_iter, end_iter, label)] for curriculum shading.'''
        spans, prev = [], 0
        for until, _tiers, label in CURRICULUM:
            spans.append((prev + 1, until, label)); prev = until
        return spans

## Model and objective builders

Two helpers used by the fine-tune run below. `build_edit_run` composes the
hydra config, builds a **fresh** trim model + cost module, selects which of
the three cost components (`demand` / `route` / `connectivity`) are active,
and enables the optional adjustment-degree shaping through `adj_weight`.
`train_edit_run` wraps `train_lc_improvement_cfg` and returns the in-memory
history. Adjustment conditioning stays off, so the fine-tuned actor keeps the
same architecture as the RTT+connectivity checkpoint.


In [6]:
def build_edit_run(run_name, disabled_components, vary_weights=True,
                   critic_overrides=None, route_time_weight=None, adj_weight=0.0,
                   adj_target=None, adj_objective=None, adj_gap=None, adj_mode=None):
    """Compose cfg + build a fresh trim model/cost module for one training run."""
    critic_overrides = list(CRITIC_OVERRIDES if critic_overrides is None
                            else critic_overrides)
    adj_target = ADJ_TRAIN_TARGET if adj_target is None else adj_target
    adj_objective = ADJ_TRAIN_OBJECTIVE if adj_objective is None else adj_objective
    adj_gap = ADJ_GAP if adj_gap is None else adj_gap
    adj_mode = ADJ_MODE if adj_mode is None else adj_mode
    overrides = [
        "model=bestsofar_feb2023_trim",
        "model.route_generator.kwargs.serial_halting=True",
        "model.route_generator.kwargs.allow_trim_below_min=true",
        f"++run_name={run_name}", "++experiment.logdir=null",
        f"++experiment.cost_function.kwargs.connectivity_mode={CONNECTIVITY_MODE}",
        f"++adjustment_degree_weight={float(adj_weight)}",
        f"++adjustment_degree_target={float(adj_target)}",
        f"++adjustment_degree_objective={adj_objective}",
        f"++adjustment_degree_gap={float(adj_gap)}",
        f"++adjustment_degree_mode={adj_mode}",
        "++adjustment_conditioning=false",
        "++adjustment_condition_current=false",
        "++adjustment_condition_weight=false",
        f"++entropy_weight={float(ENTROPY_WEIGHT)}",
        f"++force_nonhalt_first_step_until_iter={int(FORCE_NONHALT_UNTIL_ITER)}",
        f"++positive_only_trim_reward={str(POSITIVE_ONLY_TRIM_REWARD).lower()}",
        f"++zero_trim_reward={str(ZERO_TRIM_REWARD).lower()}",
    ] + critic_overrides
    with initialize_config_dir(config_dir=str(CFG_DIR), version_base=None):
        cfg = compose(config_name="ppo_50nodes.yaml", overrides=overrides)
    _, run_name, _, cost_obj, model = lrnu.process_standard_experiment_cfg(
        cfg, run_name_prefix="improvement_")
    cost_obj.ignore_stops_oob = True
    cost_obj.set_enabled_components(disabled_components=disabled_components or None)
    if vary_weights:
        cost_obj.variable_weights = True
        cost_obj.pp_fraction = 0.0
        cost_obj.op_fraction = OP_FRACTION
        cost_obj.mcw_fraction = MCW_FRACTION
    else:
        cost_obj.variable_weights = False
    if route_time_weight is not None:
        cost_obj.route_time_weight = float(route_time_weight)
    best_path = EDIT_MODEL_WEIGHTS_DIR / f"{run_name}.pt"
    print(f"run_name={run_name} | enabled={list(cost_obj.enabled_component_names)} | "
          f"variable_weights={cost_obj.variable_weights} | edge_dim={model.edge_feat_dim}")
    print(f"  connectivity_mode={cost_obj.connectivity_mode}")
    print(f"  trim reward: zero_trim={cfg.get('zero_trim_reward')} "
          f"pos_only={cfg.get('positive_only_trim_reward')} | "
          f"critic_norm={cfg.get('critic_normalize_returns')}")
    print(f"  adj shaping: W={cfg.get('adjustment_degree_weight')} "
          f"target={cfg.get('adjustment_degree_target')} "
          f"objective={cfg.get('adjustment_degree_objective')} "
          f"mode={cfg.get('adjustment_degree_mode')} gap={cfg.get('adjustment_degree_gap')}")
    return cfg, cost_obj, model, run_name, best_path


def _pad_route_tensor(t, n_routes, width):
    if t.ndim == 2:
        t = t.unsqueeze(0)
    out = torch.full((t.shape[0], n_routes, width), -1,
                     dtype=t.dtype, device=t.device)
    nr = min(n_routes, t.shape[1])
    w = min(width, t.shape[2])
    out[:, :nr, :w] = t[:, :nr, :w]
    return out


def _adj_penalty_from_routes(routes, reference, cost_obj, *, weight, target,
                             objective, gap, mode):
    n_routes = max(int(routes.shape[1]), int(reference.shape[1]))
    width = max(int(routes.shape[2]), int(reference.shape[2]))
    routes = _pad_route_tensor(routes, n_routes, width)
    reference = _pad_route_tensor(reference, n_routes, width)
    adj = get_adjustment_degrees(
        routes, reference, cost_obj.symmetric_routes, gap=gap, mode=mode)
    if objective == "cap":
        pen = (adj - float(target)).clamp(min=0.0)
    elif objective == "cap_sq":
        pen = (adj - float(target)).clamp(min=0.0) ** 2
    elif objective == "target":
        pen = (adj - float(target)).abs()
    else:
        pen = adj
    return float(weight) * pen.mean(dim=1), adj.mean(dim=1)


# train_lc_improvement_cfg validates and saves best checkpoints through the
# module-level evaluate_lc_improvement. Patch it locally so validation is scored
# with the same adj cap penalty as the PPO reward and later BCO experiments.
import connectpt.routes_generator.improvement_learning as _il_mod
if not hasattr(_il_mod, "_paper_combined_raw_evaluate_lc_improvement"):
    _il_mod._paper_combined_raw_evaluate_lc_improvement = _il_mod.evaluate_lc_improvement
_RAW_EVALUATE_LC_IMPROVEMENT = _il_mod._paper_combined_raw_evaluate_lc_improvement


def evaluate_lc_improvement_adj_aware(model, cost_obj, graphs, seed_routes, indices,
                                      device, min_route_len, max_route_len,
                                      batch_size=8, force_nonhalt_first_step=False,
                                      max_route_edit_steps=None,
                                      max_trim_actions_per_route=1,
                                      return_action_stats=False,
                                      target_n_routes=None,
                                      return_best_routes=False,
                                      adjustment_target=None,
                                      adjustment_weight=None,
                                      adjustment_use_current=False,
                                      adjustment_gap=0.1,
                                      adjustment_mode="paper"):
    result = _RAW_EVALUATE_LC_IMPROVEMENT(
        model, cost_obj, graphs, seed_routes, indices, device,
        min_route_len, max_route_len, batch_size=batch_size,
        force_nonhalt_first_step=force_nonhalt_first_step,
        max_route_edit_steps=max_route_edit_steps,
        max_trim_actions_per_route=max_trim_actions_per_route,
        return_action_stats=return_action_stats,
        target_n_routes=target_n_routes,
        return_best_routes=return_best_routes,
        adjustment_target=adjustment_target,
        adjustment_weight=adjustment_weight,
        adjustment_use_current=adjustment_use_current,
        adjustment_gap=adjustment_gap,
        adjustment_mode=adjustment_mode)

    eval_weights = cost_obj.get_weights(device)
    raw_seed_costs, raw_final_costs = [], []
    seed_adj_pens, final_adj_pens = [], []
    seed_adjs, final_adjs = [], []

    batch_splits = list(indices.split(batch_size))
    for batch_indices, final_routes_cpu in zip(batch_splits, result["routes"]):
        graph_batch, route_batch = make_improvement_batch(
            graphs, seed_routes, batch_indices, device, training=False,
            target_n_routes=target_n_routes)
        final_routes = final_routes_cpu.to(device)
        if final_routes.ndim == 2:
            final_routes = final_routes.unsqueeze(0)

        seed_state = RouteGenBatchState(
            graph_batch, cost_obj, route_batch.shape[1],
            min_route_len, max_route_len,
            cost_weights=_il_mod._clone_cost_weights(eval_weights))
        seed_state.add_new_routes(route_batch)
        final_state = RouteGenBatchState(
            graph_batch, cost_obj, final_routes.shape[1],
            min_route_len, max_route_len,
            cost_weights=_il_mod._clone_cost_weights(eval_weights))
        final_state.add_new_routes(final_routes)

        raw_seed_costs.append(cost_obj(seed_state).cost.detach().cpu())
        raw_final_costs.append(cost_obj(final_state).cost.detach().cpu())
        sp, sa = _adj_penalty_from_routes(
            route_batch, route_batch, cost_obj,
            weight=ADJ_TRAIN_WEIGHT, target=ADJ_TRAIN_TARGET,
            objective=ADJ_TRAIN_OBJECTIVE, gap=ADJ_GAP, mode=ADJ_MODE)
        fp, fa = _adj_penalty_from_routes(
            final_routes, route_batch, cost_obj,
            weight=ADJ_TRAIN_WEIGHT, target=ADJ_TRAIN_TARGET,
            objective=ADJ_TRAIN_OBJECTIVE, gap=ADJ_GAP, mode=ADJ_MODE)
        seed_adj_pens.append(sp.detach().cpu())
        final_adj_pens.append(fp.detach().cpu())
        seed_adjs.append(sa.detach().cpu())
        final_adjs.append(fa.detach().cpu())

    raw_seed_costs = torch.cat(raw_seed_costs)
    raw_final_costs = torch.cat(raw_final_costs)
    seed_adj_pens = torch.cat(seed_adj_pens)
    final_adj_pens = torch.cat(final_adj_pens)
    seed_adjs = torch.cat(seed_adjs)
    final_adjs = torch.cat(final_adjs)
    adj_seed_costs = raw_seed_costs + seed_adj_pens
    adj_final_costs = raw_final_costs + final_adj_pens

    result.update({
        "raw_seed_cost": raw_seed_costs.mean().item(),
        "raw_final_cost": raw_final_costs.mean().item(),
        "raw_delta": (raw_seed_costs - raw_final_costs).mean().item(),
        "seed_adjustment_penalty": seed_adj_pens.mean().item(),
        "final_adjustment_penalty": final_adj_pens.mean().item(),
        "seed_adjustment_degree": seed_adjs.mean().item(),
        "final_adjustment_degree": final_adjs.mean().item(),
        "adjustment_penalty_delta": (seed_adj_pens - final_adj_pens).mean().item(),
        "adjustment_degree_delta": (seed_adjs - final_adjs).mean().item(),
        "seed_cost": adj_seed_costs.mean().item(),
        "final_cost": adj_final_costs.mean().item(),
        "delta": (adj_seed_costs - adj_final_costs).mean().item(),
        "win_rate": (adj_final_costs < adj_seed_costs).float().mean().item(),
    })
    return result


_il_mod.evaluate_lc_improvement = evaluate_lc_improvement_adj_aware


def train_edit_run(model, cost_obj, cfg, run_name, best_path,
                   n_iterations, batch_size, train_indices, val_indices,
                   curriculum_fn=None, checkpoint=None):
    """Train one edit model and return its in-memory history DataFrame."""
    result = train_lc_improvement_cfg(
        model=model, cost_obj=cost_obj, graphs=graphs, seed_routes=seed_routes,
        device=device, cfg=cfg, output_dir=MODEL_OUTPUTS_DIR, run_name=run_name,
        train_fraction=TRAIN_FRACTION, batch_size=batch_size,
        min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN, seed=SPLIT_SEED,
        max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
        max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE,
        target_n_routes=TARGET_N_ROUTES,
        train_indices=train_indices, val_indices=val_indices,
        best_model_path=best_path, n_iterations=n_iterations,
        force_nonhalt_first_step=FORCE_NONHALT_FIRST_STEP,
        curriculum_fn=curriculum_fn,
        history_checkpoint_path=checkpoint,
    )
    df = pd.DataFrame(result["history"])
    save_table(df, f"{run_name}_training_history")
    print(f"history rows={len(df)}; best -> {best_path}")
    return df


print("builders ready: build_edit_run(), train_edit_run()")
print("adj-aware validation installed: val cost/delta/win_rate include the paper-mode cap penalty")


builders ready: build_edit_run(), train_edit_run()
adj-aware validation installed: val cost/delta/win_rate include the paper-mode cap penalty


## 100-epoch fine-tune - route + connectivity + adj (W=10)

This run starts from the final-experiments RTT + connectivity checkpoint,
keeps `route` and `connectivity` active at 0.5/0.5 (demand off), and adds
adjustment-degree cap reward shaping with W=10, target=0.2, and paper-mode alignment. The budget is 100
epochs over the current 500-graph copy-redundancy curriculum, batch 4, with
balanced per-tier validation monitoring. The resulting checkpoint is saved
under a new `adjcap` run name and is used by the evaluation cells below.

If `RESUME_FROM_CHECKPOINT=True`, this cell reloads the adj-finetune output
first; otherwise it initializes from `BASE_MODEL_PATH`.


In [7]:
if RUN_TRAINING:
    # Fine-tune: route + connectivity (0.5/0.5) + adj penalty (W=ADJ_TRAIN_WEIGHT); demand off.
    cfg, cost_obj, model, run_name, BEST_MODEL_PATH = build_edit_run(
        run_name=RUN_NAME,
        disabled_components=DISABLED_COST_COMPONENTS,
        vary_weights=VARY_WEIGHTS,
        route_time_weight=ROUTE_W,
        adj_weight=ADJ_TRAIN_WEIGHT,
        adj_target=ADJ_TRAIN_TARGET,
        adj_objective=ADJ_TRAIN_OBJECTIVE,
        adj_gap=ADJ_GAP, adj_mode=ADJ_MODE)
    cost_obj.median_connectivity_weight = float(CONN_W)   # fix conn weight (demand off)
    print(f"train objective: route={ROUTE_W} conn={CONN_W} demand=off | "
          f"adj W={ADJ_TRAIN_WEIGHT} target={ADJ_TRAIN_TARGET} "
          f"objective={ADJ_TRAIN_OBJECTIVE} mode={ADJ_MODE} gap={ADJ_GAP}")

    # Resume the adj fine-tune if requested; otherwise initialize from the
    # final-experiments RTT+connectivity checkpoint and save to a new run name.
    if RESUME_FROM_CHECKPOINT and BEST_MODEL_PATH.exists():
        model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
        print(f"resumed adj-finetune weights <- {BEST_MODEL_PATH}")
    elif BASE_MODEL_PATH.exists():
        model.load_state_dict(torch.load(BASE_MODEL_PATH, map_location=device))
        print(f"initialized adj-finetune from parent PRESERVED checkpoint <- {BASE_MODEL_PATH}")
    elif REQUIRE_BASE_MODEL:
        raise FileNotFoundError(
            "No base RTT+connectivity model found. Checked: "
            + str(BASE_MODEL_PATH)
        )
    elif RESUME_FROM_CHECKPOINT:
        print(f"RESUME requested but no checkpoint at {BEST_MODEL_PATH}; training from scratch")
    else:
        print("base checkpoint missing; training from scratch")

    _t_train0 = _time.perf_counter()
    history_df = train_edit_run(
        model, cost_obj, cfg, FULL_HISTORY_RUN, BEST_MODEL_PATH,
        n_iterations=N_ITERATIONS, batch_size=BATCH_SIZE,
        train_indices=TRAIN_INDICES, val_indices=MONITOR_VAL_INDICES,
        curriculum_fn=(curriculum_fn if USE_CURRICULUM else None),
        checkpoint=FULL_HISTORY_CHECKPOINT)
    print(f"continuation history -> {FULL_HISTORY_RUN}_training_history "
          f"(partial: {FULL_HISTORY_CHECKPOINT.name})")
    TIMING["epoch_s"] = (_time.perf_counter() - _t_train0) / max(1, N_ITERATIONS)
    print(f"[timing] training: {TIMING['epoch_s']:.2f} s/epoch over {N_ITERATIONS} epochs")

## Train the edit model (cumulative curriculum)

Fine-tune the edit model over the cumulative curriculum (duplicate -> boundary
-> mixed -> covered -> clean). Training history is checkpointed to a partial CSV;
on resume the prior history is prepended.

In [8]:
if RUN_TRAINING:
    if "pd" not in globals():
        import pandas as pd
    if "plt" not in globals():
        import matplotlib.pyplot as plt
    from pathlib import Path

    # --- stitch history: prior part(s) (e.g. machine-1) + this run's continuation ---
    _parts, _labels = [], []
    for _f in (PRIOR_HISTORY_FILES if "PRIOR_HISTORY_FILES" in globals() else []):
        _f = Path(_f)
        if _f.exists():
            _parts.append(pd.read_csv(_f)); _labels.append(f"{_f.name}({len(_parts[-1])})")
    if "history_df" in globals():
        _parts.append(history_df.copy()); _labels.append(f"in-memory({len(history_df)})")
    elif "FULL_HISTORY_CHECKPOINT" in globals() and Path(FULL_HISTORY_CHECKPOINT).exists():
        _parts.append(pd.read_csv(FULL_HISTORY_CHECKPOINT))
        _labels.append(f"{Path(FULL_HISTORY_CHECKPOINT).name}({len(_parts[-1])})")
    if not _parts:
        raise FileNotFoundError("No history found. Train at least one epoch first.")
    h = pd.concat(_parts, ignore_index=True)
    h["epoch"] = range(1, len(h) + 1)          # continuous axis across stitched parts
    print(f"history stitched: {' + '.join(_labels)} => {len(h)} epochs")

    def _num(col):
        return pd.to_numeric(h[col], errors="coerce") if col in h.columns else None

    # curriculum stage spans derived from the stitched data (robust to stitching)
    if "curriculum_stage" in h.columns and h["curriculum_stage"].notna().any():
        _spans = []
        for _epoch, _label in h[["epoch", "curriculum_stage"]].dropna().itertuples(index=False, name=None):
            _epoch = int(_epoch)
            if _spans and _spans[-1][2] == _label:
                _spans[-1] = (_spans[-1][0], _epoch, _label)
            else:
                _spans.append((_epoch, _epoch, _label))
    elif "stage_spans" in globals():
        _spans = stage_spans()
    else:
        _spans = []
    _colors = ["#eaf3ff", "#eafbea", "#fff6e6", "#fdeaea", "#f0eaff"]
    def _shade(ax):
        for k, (s, e, lab) in enumerate(_spans):
            ax.axvspan(s, e, color=_colors[k % len(_colors)], alpha=0.6, zorder=0)
            ax.axvline(s, color="gray", lw=0.6, ls=":")

    fig, ax = plt.subplots(2, 3, figsize=(17, 8), constrained_layout=True)
    panels = [("train_reward_mean","train reward"), ("val_delta","val cost delta (+=улучш.)"),
              ("val_win_rate","val win rate"), ("train_action_avg_actions_per_route","avg edits/route"),
              ("val_component_delta_route","val route delta"),
              ("val_component_delta_connectivity","val conn delta")]
    for a,(col,title) in zip(ax.flat, panels):
        _shade(a); y=_num(col)
        if y is not None and y.notna().any():
            a.plot(h["epoch"], y, marker="o", ms=2, color="tab:blue", zorder=3)
        a.axhline(0,color="k",lw=0.7); a.set_title(title); a.set_xlabel("epoch"); a.grid(alpha=0.2)
    # подписи стадий сверху
    for s,e,lab in _spans:
        ax[0,0].text((s+e)/2, ax[0,0].get_ylim()[1], lab, ha="center", va="bottom", fontsize=8)
    fig.suptitle("Actor curves + curriculum stages (заливка = стадия; история сшита)",
                 fontsize=13, fontweight="bold")
    plt.show(); plt.close(fig)

## Critic diagnostics (per-component critic MSE / explained variance)

In [9]:
if RUN_TRAINING:
    crit_cols = [c for c in h.columns if "critic" in c.lower()]
    print("critic columns:", crit_cols)
    if crit_cols:
        n=len(crit_cols)
        fig, ax = plt.subplots(1, n, figsize=(5*n, 4), squeeze=False, constrained_layout=True)
        for a, col in zip(ax[0], crit_cols):
            _shade(a); y=_num(col)
            if y is not None and y.notna().any():
                a.plot(h["epoch"], y, marker="o", ms=2, color="tab:orange", zorder=3)
            a.set_title(col, fontsize=9); a.set_xlabel("epoch"); a.grid(alpha=0.2)
            if "explained" in col: a.axhline(0, color="k", lw=0.7)
        fig.suptitle("Critic metrics + curriculum stages", fontsize=13, fontweight="bold")
        plt.show(); plt.close(fig)
        display(h[["epoch","curriculum_stage"]+crit_cols].iloc[::max(1,len(h)//15)].round(4))

## Evaluate balanced policy by tier

The table reports before/after route-level redundancy, `ATT`, `RTT`,
connectivity, and demand percentages by transfer bucket (`d0`, `d1`, `d2`,
`d_un`). `Adj(current, seed)` remains available only in the example plots.

In [10]:
if RUN_TRAINING:
    _required_eval_globals = (
        "TIERS", "EVAL_N_PER_TIER", "_val_by_tier", "BALANCED_EVAL_WEIGHTS",
        "graphs", "seed_routes", "model", "cost_obj", "device",
        "TARGET_N_ROUTES", "MIN_ROUTE_LEN", "MAX_ROUTE_LEN",
        "MAX_ROUTE_EDIT_STEPS", "MAX_TRIM_ACTIONS_PER_ROUTE",
        "_redundancy_stats", "make_improvement_batch", "rollout_lc_improvement",
        "get_batch_tensor_from_routes", "torch", "tqdm",
    )
    _missing_eval_globals = [name for name in _required_eval_globals if name not in globals()]
    if _missing_eval_globals:
        raise RuntimeError(
            "Balanced eval needs initialized config, dataset split, and model. "
            "Run the notebook cells from imports through training first. Missing: "
            + ", ".join(_missing_eval_globals)
        )

    def _redun_t(routes_2d):
        return _redundancy_stats(routes_2d)["redundancy"]


    def _mean_metric(result, key):
        value = result.get_metrics()[key]
        return float(value.detach().float().mean().item())


    val_by_tier = {
        tier: indices[:EVAL_N_PER_TIER]
        for tier, indices in _val_by_tier.items()
    }

    base_w = cost_obj.get_weights(device)
    def mkw(route_weight, conn_weight):
        weights = {key: (value.clone() if torch.is_tensor(value) else value)
                   for key, value in base_w.items()}
        weights["demand_time_weight"] = torch.as_tensor(0.0, device=device)
        weights["route_time_weight"] = torch.as_tensor(float(route_weight), device=device)
        weights["median_connectivity_weight"] = torch.as_tensor(float(conn_weight), device=device)
        return weights


    rows = []
    visual_examples = {}
    conn_metric_key = ("median_connectivity_weighted"
                       if cost_obj.use_weighted_connectivity
                       else "median_connectivity")
    transfer_metric_keys = {
        "d0": "$d_0$", "d1": "$d_1$", "d2": "$d_2$", "d_un": "$d_{un}$",
    }
    model.eval()
    weights = mkw(*BALANCED_EVAL_WEIGHTS)
    for tier in TIERS:
        idxs = val_by_tier[tier]
        if not idxs:
            continue
        metrics = {
            "redun_before": [], "redun_after": [],
            "ATT_before": [], "ATT_after": [],
            "RTT_before": [], "RTT_after": [],
            "CONN_before": [], "CONN_after": [],
            "d0_before": [], "d0_after": [],
            "d1_before": [], "d1_after": [],
            "d2_before": [], "d2_after": [],
            "d_un_before": [], "d_un_after": [],
        }
        for gi in tqdm(idxs, desc=f"eval balanced/{tier}", leave=False):
            graph_batch, route_batch = make_improvement_batch(
                graphs, seed_routes, torch.tensor([gi]), device,
                training=False, target_n_routes=TARGET_N_ROUTES)
            with torch.no_grad():
                output = rollout_lc_improvement(
                    model, cost_obj, graph_batch, route_batch,
                    MIN_ROUTE_LEN, MAX_ROUTE_LEN,
                    greedy=True, cost_weights=weights,
                    max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
                    max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE)
            final_state, seed_result, final_result = output[:3]
            improved = get_batch_tensor_from_routes(
                final_state.routes, device, max_route_len=route_batch.shape[-1])
            metrics["redun_before"].append(_redun_t(route_batch[0]))
            metrics["redun_after"].append(_redun_t(improved[0]))
            metrics["ATT_before"].append(_mean_metric(seed_result, "ATT"))
            metrics["ATT_after"].append(_mean_metric(final_result, "ATT"))
            metrics["RTT_before"].append(_mean_metric(seed_result, "RTT"))
            metrics["RTT_after"].append(_mean_metric(final_result, "RTT"))
            metrics["CONN_before"].append(_mean_metric(seed_result, conn_metric_key))
            metrics["CONN_after"].append(_mean_metric(final_result, conn_metric_key))
            for metric_name, metric_key in transfer_metric_keys.items():
                metrics[f"{metric_name}_before"].append(_mean_metric(seed_result, metric_key))
                metrics[f"{metric_name}_after"].append(_mean_metric(final_result, metric_key))

            if tier not in visual_examples:
                nr = min(improved.shape[1], route_batch.shape[1])
                width = min(improved.shape[-1], route_batch.shape[-1])
                adj = get_adjustment_degrees(
                    improved[:, :nr, :width], route_batch[:, :nr, :width],
                    cost_obj.symmetric_routes, gap=ADJ_GAP, mode=ADJ_MODE
                ).mean().item()
                visual_examples[tier] = {
                    "graph_index": gi,
                    "seed": route_batch[0].detach().cpu(),
                    "improved": improved[0].detach().cpu(),
                    "Adj": adj,
                    "redun_before": metrics["redun_before"][-1],
                    "redun_after": metrics["redun_after"][-1],
                    "ATT_before": metrics["ATT_before"][-1],
                    "ATT_after": metrics["ATT_after"][-1],
                    "RTT_before": metrics["RTT_before"][-1],
                    "RTT_after": metrics["RTT_after"][-1],
                    "CONN_before": metrics["CONN_before"][-1],
                    "CONN_after": metrics["CONN_after"][-1],
                }

        means = {key: float(np.mean(values)) for key, values in metrics.items()}
        rows.append({
            "tier": tier, "n": len(idxs),
            "redun_before": means["redun_before"],
            "redun_after": means["redun_after"],
            "ATT_before": means["ATT_before"], "ATT_after": means["ATT_after"],
            "RTT_before": means["RTT_before"], "RTT_after": means["RTT_after"],
            "CONN_before": means["CONN_before"], "CONN_after": means["CONN_after"],
            "d0_before": means["d0_before"], "d0_after": means["d0_after"],
            "d1_before": means["d1_before"], "d1_after": means["d1_after"],
            "d2_before": means["d2_before"], "d2_after": means["d2_after"],
            "d_un_before": means["d_un_before"], "d_un_after": means["d_un_after"],
        })

    eval_df = pd.DataFrame(rows).round(4)
    display(eval_df)
    save_table(eval_df, f"{RUN_NAME}_eval_by_tier")
    print("ATT/RTT/CONN are minutes; d0/d1/d2/d_un are demand percentages by transfer bucket.")

## Visual validation examples

For the balanced preference vector, show one seed and the corresponding edited
network from every tier.  The right-hand panels emphasize removed and added
segments relative to the corrupted seed.

In [11]:
if RUN_TRAINING:
    if not visual_examples:
        print("Run the evaluation cell first.")
    else:
        tiers_to_plot = [tier for tier in TIERS if tier in visual_examples]
        fig, axes = plt.subplots(
            len(tiers_to_plot), 2,
            figsize=(18, 7 * len(tiers_to_plot)),
            squeeze=False, constrained_layout=True)
        for row_idx, tier in enumerate(tqdm(tiers_to_plot, desc="render tiers")):
            example = visual_examples[tier]
            graph = graphs[example["graph_index"]]
            route_plots.plot_plain_route_set(
                axes[row_idx, 0], example["seed"], graph,
                title=f"{tier}: corrupted seed (graph {example['graph_index']})",
                subtitle=(f"redun={example['redun_before']:.3f}; "
                          f"ATT={example['ATT_before']:.2f}; RTT={example['RTT_before']:.2f}; "
                          f"CONN={example['CONN_before']:.2f}"))
            route_plots.plot_route_diff(
                axes[row_idx, 1], example["improved"], example["seed"], graph,
                title=f"{tier}: edited network vs seed",
                subtitle=(f"redun {example['redun_before']:.3f}->{example['redun_after']:.3f}; "
                          f"Adj={example['Adj']:.3f}\n"
                          f"ATT {example['ATT_before']:.2f}->{example['ATT_after']:.2f}; "
                          f"RTT {example['RTT_before']:.2f}->{example['RTT_after']:.2f}; "
                          f"CONN {example['CONN_before']:.2f}->{example['CONN_after']:.2f}"))
        fig.suptitle("Balanced validation: copy/subcopy corruption repair", fontsize=15, fontweight="bold")
        plt.show()
        plt.close(fig)

## Agent action GIFs

Run the greedy balanced policy step by step for one graph from every tier.
Each GIF starts from the corrupted seed and adds a frame after every agent
action (`extend`, `trim_start`, `trim_end`, or `halt`) with its reward and cost.

In [12]:
# GIF_OUTPUT_DIR = MODEL_OUTPUTS_DIR / "agent_action_gifs"
# GIF_FPS = 2
# GIF_FORCE_NONHALT_FIRST_STEP = False  # match the balanced eval rollout
# ACTION_NAMES = {
#     ROUTE_ACTION_EXTEND: "extend", ROUTE_ACTION_TRIM_START: "trim_start",
#     ROUTE_ACTION_TRIM_END: "trim_end", ROUTE_ACTION_HALT: "halt",
# }


# def _is_trim_action(action_kind):
#     return int(action_kind) in (ROUTE_ACTION_TRIM_START, ROUTE_ACTION_TRIM_END)


# def _action_text(action_kind, action):
#     action_kind = int(action_kind)
#     name = ACTION_NAMES[action_kind]
#     if action_kind == ROUTE_ACTION_HALT:
#         return name
#     if _is_trim_action(action_kind):
#         return f"{name}(position={int(action[0])})"
#     return f"{name}({int(action[0])} -> {int(action[1])})"


# def _put_route(routes, route_idx, route):
#     routes = routes.clone()
#     routes[:, route_idx] = -1
#     route = route[route > -1].to(routes.device)
#     if route.numel():
#         route_tensor = get_batch_tensor_from_routes(
#             [[route]], routes.device, max_route_len=routes.shape[-1])
#         routes[:, route_idx] = route_tensor[:, 0]
#     return routes


# def _collect_agent_action_frames(graph_index):
#     graph_batch, route_batch = make_improvement_batch(
#         graphs, seed_routes, torch.tensor([graph_index]), device,
#         training=False, target_n_routes=TARGET_N_ROUTES)
#     weights = mkw(*BALANCED_EVAL_WEIGHTS)
#     reward_scale = float(getattr(cfg, "reward_scale", 1.0))
#     diff_reward = bool(getattr(cfg, "diff_reward", True))
#     incumbent_reward = bool(getattr(cfg, "incumbent_reward", False))
#     zero_trim_reward = bool(getattr(cfg, "zero_trim_reward", False))
#     positive_only_trim_reward = bool(getattr(cfg, "positive_only_trim_reward", False))
#     edit_step_penalty = float(getattr(cfg, "edit_step_penalty", 0.0))
#     forced_halt_penalty = float(getattr(cfg, "forced_halt_penalty", 0.0))
#     context_len = max(
#         int(route_batch.shape[-1]), int(MAX_ROUTE_LEN),
#         int(graph_batch[STOP_KEY].num_nodes))
#     working_routes = torch.full(
#         (1, route_batch.shape[1], context_len), -1,
#         dtype=route_batch.dtype, device=device)
#     working_routes[..., :route_batch.shape[-1]] = route_batch
#     display_routes = working_routes.clone()
#     frames = [{"routes": display_routes[0].detach().cpu(), "label": "corrupted seed"}]
#     rows = []

#     model.eval()
#     with torch.no_grad():
#         for route_idx in range(route_batch.shape[1]):
#             context_routes = working_routes.clone()
#             context_routes[:, route_idx] = -1
#             route_state = _make_route_context_state(
#                 cost_obj, graph_batch, working_routes, route_idx,
#                 MIN_ROUTE_LEN, MAX_ROUTE_LEN, weights,
#                 invalid_directly_connected=not bool((context_routes >= 0).any().item()))
#             context_counts = route_state.n_finished_routes.detach().clone()
#             route_state = model.setup_planning(route_state)
#             prev_cost = cost_obj(route_state).cost.detach().clone()
#             trim_count = 0

#             for step_idx in range(MAX_ROUTE_EDIT_STEPS + 1):
#                 if route_state.is_done().all():
#                     break
#                 force_halt = step_idx >= MAX_ROUTE_EDIT_STEPS
#                 trim_allowed = (MAX_TRIM_ACTIONS_PER_ROUTE is None or
#                                 trim_count < int(MAX_TRIM_ACTIONS_PER_ROUTE))
#                 if force_halt:
#                     step_kinds = torch.full((1,), ROUTE_ACTION_HALT, dtype=torch.long, device=device)
#                     step_actions = torch.full((1, 2), -1, dtype=torch.long, device=device)
#                 else:
#                     step_kinds, step_actions, _, _ = model.step_route_action(
#                         route_state, greedy=True,
#                         allow_halt=not (GIF_FORCE_NONHALT_FIRST_STEP and step_idx == 0),
#                         allow_trim_start=trim_allowed, allow_trim_end=trim_allowed)

#                 cost_before = float(prev_cost.cpu()[0])
#                 action_kind = int(step_kinds[0].item())
#                 is_trim = _is_trim_action(action_kind)
#                 route_state.apply_route_actions(step_kinds, step_actions)
#                 planned_route = _get_planned_current_routes(
#                     route_state, working_routes[:, route_idx], context_counts)[0]
#                 frame_routes = _put_route(display_routes, route_idx, planned_route)
#                 new_cost = cost_obj(route_state).cost.detach().clone()
#                 if incumbent_reward:
#                     step_reward = torch.clamp_min(prev_cost - new_cost, 0) * reward_scale
#                 elif diff_reward:
#                     step_reward = (prev_cost - new_cost) * reward_scale
#                 else:
#                     step_reward = torch.zeros_like(new_cost)
#                     if action_kind == ROUTE_ACTION_HALT:
#                         step_reward = -new_cost * reward_scale
#                 if positive_only_trim_reward and is_trim:
#                     step_reward = step_reward.clamp_min(0)
#                 elif zero_trim_reward and is_trim:
#                     step_reward = torch.zeros_like(step_reward)
#                 if action_kind != ROUTE_ACTION_HALT:
#                     step_reward = step_reward - edit_step_penalty
#                 if force_halt:
#                     step_reward = step_reward - forced_halt_penalty
#                 if not (zero_trim_reward and not positive_only_trim_reward and is_trim):
#                     prev_cost = new_cost
#                 trim_count += int(is_trim)

#                 label = (_action_text(action_kind, step_actions[0]) +
#                          f" | reward={float(step_reward.cpu()[0]):+.4f}" +
#                          f" | cost={float(new_cost.cpu()[0]):.4f}")
#                 frames.append({"routes": frame_routes[0].detach().cpu(), "label": label})
#                 rows.append({
#                     "route": route_idx, "step": step_idx + 1,
#                     "action": _action_text(action_kind, step_actions[0]),
#                     "action_kind": ACTION_NAMES[action_kind],
#                     "cost_before": cost_before, "cost_after": float(new_cost.cpu()[0]),
#                     "reward": float(step_reward.cpu()[0]), "forced_halt": force_halt,
#                 })
#                 if action_kind == ROUTE_ACTION_HALT:
#                     break

#             final_route = _get_planned_current_routes(
#                 route_state, working_routes[:, route_idx], context_counts)[0]
#             display_routes = _put_route(display_routes, route_idx, final_route)
#             working_routes = _put_route(working_routes, route_idx, final_route)
#     return frames, pd.DataFrame(rows)


# def save_agent_action_gif(tier, example):
#     frames, steps_df = _collect_agent_action_frames(example["graph_index"])
#     graph = graphs[example["graph_index"]]
#     fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)

#     def _draw(frame_idx):
#         ax.clear()
#         frame = frames[frame_idx]
#         route_plots.plot_route_diff(
#             ax, frame["routes"], example["seed"], graph,
#             title=f"{tier}: greedy balanced actions (graph {example['graph_index']})",
#             subtitle=f"frame {frame_idx + 1}/{len(frames)} | {frame['label']}")
#         return []

#     GIF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#     gif_path = GIF_OUTPUT_DIR / f"{RUN_NAME}_{tier}_agent_actions.gif"
#     animation = FuncAnimation(
#         fig, _draw, frames=len(frames), interval=1000 / GIF_FPS, blit=False)
#     try:
#         animation.save(gif_path, writer=PillowWriter(fps=GIF_FPS))
#         plt.close(fig)
#         display(Image(filename=str(gif_path)))
#     except Exception as exc:
#         plt.close(fig)
#         print(f"{tier}: could not save GIF with PillowWriter: {exc}")
#         fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
#         _draw(len(frames) - 1)
#         plt.show(); plt.close(fig)
#         gif_path = None
#     return gif_path, steps_df


# if not visual_examples:
#     print("Run the evaluation cell first.")
# else:
#     saved_gifs, gif_steps_by_tier = {}, {}
#     for tier in tqdm([tier for tier in TIERS if tier in visual_examples], desc="render GIFs"):
#         gif_path, steps_df = save_agent_action_gif(tier, visual_examples[tier])
#         saved_gifs[tier] = gif_path
#         gif_steps_by_tier[tier] = steps_df
#         save_table(steps_df, f"{RUN_NAME}_{tier}_agent_action_steps")
#         print(f"{tier}: {len(steps_df) + 1} frames -> {gif_path}")
#         display(steps_df.groupby("action_kind")["reward"].agg(["count", "sum", "mean"]).round(4))

---
# PART 2 - BCO experiments (E1-E6)

Uses the model trained above (`BEST_MODEL_PATH`). All baselines are re-run live on a single seed with small budgets.

## Configuration

`SMOKE=True` -> Mandl only, 1 seed, tiny iteration budgets (just checks the
pipeline builds tables/figures). `SMOKE=False` runs Mandl and Mumford0-3
with the 25-30 hour budgeted settings below.


In [13]:
SMOKE = False
QUICK = False  # minimal budgets/sweeps just to verify the notebook runs end-to-end
CITIES = ["Mandl"] if SMOKE else ["Mandl", "Mumford0", "Mumford1", "Mumford2", "Mumford3"]
SEEDS = [0]                # single seed everywhere
CONNECTIVITY_MODE = "weighted_median"
# E1 (legacy ATT/RTT baselines) is NOT re-run live here -- every city is
# already computed and dumped to paper_results. Leave empty so nothing runs;
# the combined table is assembled from the saved final_main_<city>.csv files.
E1_RUN_CITIES = []          # was ["Mumford3"]; now nothing runs live in E1
# E1u (unified objective: 0.5*RTT + 0.5*WMC + adj for ALL methods) runs live.
E1U_CITIES = ["Mandl", "Mumford0", "Mumford1"]

# Legacy E1 baselines optimize ATT+RTT at alpha=0.5 (connectivity off).
BASE_WEIGHTS = dict(demand_time_weight=0.5, route_time_weight=0.5,
                    median_connectivity_weight=0.0)
# Unified objective (E1u + our model): RTT + WMC (demand off); adj is enforced
# via a cap penalty at OUR_MAIN_ADJ_TARGET, with use_weighted_connectivity=True
# so the connectivity term is the weighted mean connectivity (WMC).
OUR_WEIGHTS = dict(demand_time_weight=0.0, route_time_weight=0.5,
                   median_connectivity_weight=0.5)

# Latest fine-tuned edit model (training is OFF here): route + connectivity + adj, W=10.
OUR_MODEL_PATH = EDIT_MODEL_WEIGHTS_DIR / "improvement_lc_rttconn_adj_w10_t02_finetune100.pt"

# BCO edit/trim bees also load the edit model via eval_lib's default path.
# Point it at the same 18-dim (redundancy) checkpoint so it matches the
# bestsofar_feb2023_trim architecture.
import eval_lib.helpers as _eh
_eh.EDIT_MODEL_WEIGHTS_PATH = OUR_MODEL_PATH
# Experiment init for every city = LC construction + covered_dup tier (overridden
# to the LC generator in the helpers cell below); BENCHMARK_INIT_MODE is unused.
import eval_lib.baselines as _eb
_eb.BENCHMARK_INIT_MODE = "nx"
if QUICK:
    _eh.BCO_N_ITERATIONS = 5   # tiny BCO budget for quick sweeps

ADJ_GAP, ADJ_MODE = 0.1, "paper"
ADJ_FIXED_W = 10.0   # fixed adj penalty weight (E2 sweeps alpha x target)
OUR_MAIN_ADJ_TARGET = 0.2   # adj cap target for the unified objective (our model + E1u)
# Version B of our NBCO (edit model as the REBUILD bee) makes the trim model run a
# full rollout as a constructor -> ~250x slower per BCO iteration (258 s/it vs 1 s/it
# on Mandl, effectively hangs on Mumford). OFF by default; opt-in for Mandl-only probes.
INCLUDE_EDIT_REBUILD = False

# Early stopping OFF: every method runs its full iteration budget so the
# convergence curves below are complete (no premature termination).
EARLY_STOP_PATIENCE = None   # no early stopping (full iteration budget)

# CPU-budgeted full profiles. BCO uses the paper-scale population size
# (10 bees), with reduced-but-not-tiny iteration counts. E2 uses a separate
# BCO budget because the Pareto sweep multiplies runs by grids x models.
DEFAULT_ALGO_SETTINGS = (
    dict(sa=120, hh=120, ga=5, ga_pop=6, nsgaii=(8, 6),
         bco=8, bco_bees=10, sa_args={}, bco_args={})
    if SMOKE else
    dict(sa=15000, hh=15000, ga=500, ga_pop=20, nsgaii=(800, 100),
         bco=500, bco_bees=10, sa_args={}, bco_args={})
)

MANDL_ALGO_SETTINGS = dict(
    sa=(120 if SMOKE else 15000),
    hh=(120 if SMOKE else 15000),
    ga=(5 if SMOKE else 500),
    ga_pop=(6 if SMOKE else 20),
    nsgaii=((8, 6) if SMOKE else (800, 100)),
    bco=(8 if SMOKE else 500),
    bco_bees=10,
    sa_args=dict(initial_temp=0.08, final_temp=0.003, cooling_rate=6.5e-5,
                 reheating_threshold=300, reheating_factor=1.02),
    bco_args=dict(worse_accept_temperature=0.02, worse_accept_decay=0.985,
                  worse_accept_min_temperature=0.001,
                  worse_selection_temperature=0.02,
                  worse_selection_decay=0.985,
                  worse_selection_uniform_mix=0.10,
                  worse_selection_elite_count=2),
)

# SA explores a single network with local moves, so larger graphs need more
# iterations (Mumford1+ stayed ~stuck at the seed with 15k). Scale by city.
SA_ITERS_BY_CITY = {"Mandl": 15000, "Mumford0": 25000, "Mumford1": 40000,
                    "Mumford2": 60000, "Mumford3": 80000}
# HH random-walk init-repair removes covered_dup duplicate-edge violations; on
# Mumford1+ it took ~88k steps. Cap it -- HH penalizes residual violations in
# its optimization loop, so a near-feasible start is enough.
HH_MAX_REPAIR_ITERS = 5000

# E2 is an expensive sweep (alpha x target x models); keep iterations bounded.
E2_BCO_ITERATIONS = 120


def algo_settings(city):
    settings = dict(DEFAULT_ALGO_SETTINGS)
    if city == "Mandl":
        settings.update(MANDL_ALGO_SETTINGS)
    if not (SMOKE or QUICK):
        settings["sa"] = SA_ITERS_BY_CITY.get(city, settings["sa"])
    settings["bco_args"] = dict(DEFAULT_ALGO_SETTINGS.get("bco_args", {}),
                                 **settings.get("bco_args", {}))
    if QUICK:
        settings.update(sa=40, hh=40, ga=5, ga_pop=6, nsgaii=(10, 8),
                        bco=8, bco_bees=10)
    # SA temperature schedule: a LINEAR anneal spanning the FULL (city-specific)
    # iteration budget + periodic reheating, applied to ALL cities. Previously
    # only Mandl was tuned; Mumford used the yaml defaults with NO reheating and
    # a budget-agnostic cooling_rate, so SA could stay stuck at the seed.
    # cost_norm in the SA loop normalizes acceptance by the initial cost, so one
    # schedule transfers across cities (initial cost ~2-3 for every benchmark).
    _sa_n = int(settings["sa"]); _t0, _t1 = 0.15, 0.005
    settings["sa_args"] = dict(
        schedule="linear", initial_temp=_t0, final_temp=_t1,
        cooling_rate=(_t0 - _t1) / max(_sa_n, 1),
        reheating_threshold=max(500, _sa_n // 20), reheating_factor=1.5)
    return settings


print("cities:", CITIES)
print("our model:", OUR_MODEL_PATH.name, "| exists:", OUR_MODEL_PATH.exists())
print("connectivity_mode:", CONNECTIVITY_MODE)
print("legacy E1 weights (ATT+RTT, alpha=0.5):", BASE_WEIGHTS)
print("unified E1u weights (RTT+WMC):", OUR_WEIGHTS, "| adj cap target:", OUR_MAIN_ADJ_TARGET)
print("E2 BCO iterations:", E2_BCO_ITERATIONS)

cities: ['Mandl', 'Mumford0', 'Mumford1', 'Mumford2', 'Mumford3']
our model: improvement_lc_rttconn_adj_w10_t02_finetune100.pt | exists: True
connectivity_mode: weighted_median
legacy E1 weights (ATT+RTT, alpha=0.5): {'demand_time_weight': 0.5, 'route_time_weight': 0.5, 'median_connectivity_weight': 0.0}
unified E1u weights (RTT+WMC): {'demand_time_weight': 0.0, 'route_time_weight': 0.5, 'median_connectivity_weight': 0.5} | adj cap target: 0.2
E2 BCO iterations: 120


## Shared metrics: adjustment-degree vs seed + redundancy

In [14]:
from omegaconf import OmegaConf
from tqdm.auto import tqdm


def bco_cfg_set(cfg, **kv):
    """Set fields BCO reads from cfg (n_iterations, adjustment_degree_*)."""
    OmegaConf.set_struct(cfg, False)
    for k, v in kv.items():
        cfg[k] = v
    return cfg


def _set_cfg_value(cfg, dotted_key, value):
    """Set a nested OmegaConf value even when the composed cfg is structured."""
    target = cfg
    parts = dotted_key.split(".")
    for part in parts[:-1]:
        OmegaConf.set_struct(target, False)
        target = target[part]
    OmegaConf.set_struct(target, False)
    target[parts[-1]] = value
    return cfg


def _sa_cfg(city, run_name, n_routes, min_route_len, max_route_len):
    s = algo_settings(city)
    cfg = build_sa_cfg(run_name, n_routes, min_route_len, max_route_len,
                       n_iterations=s["sa"], early_stop_patience=EARLY_STOP_PATIENCE,
                       connectivity_mode=CONNECTIVITY_MODE)
    for key, value in s.get("sa_args", {}).items():
        _set_cfg_value(cfg, f"alg_args.{key}", value)
    return cfg


def _ga_cfg(city, run_name, n_routes, min_route_len, max_route_len):
    s = algo_settings(city)
    return build_ga_cfg(run_name, n_routes, min_route_len, max_route_len,
                        n_iterations=s["ga"], population_size=s["ga_pop"],
                        early_stop_patience=EARLY_STOP_PATIENCE,
                        connectivity_mode=CONNECTIVITY_MODE)


def _hh_cfg(city, run_name, n_routes, min_route_len, max_route_len):
    s = algo_settings(city)
    return build_hh_cfg(run_name, n_routes, min_route_len, max_route_len,
                        n_iterations=s["hh"], max_repair_iters=HH_MAX_REPAIR_ITERS,
                        early_stop_patience=EARLY_STOP_PATIENCE,
                        connectivity_mode=CONNECTIVITY_MODE)


def _variant_type1_bees(variant, n_bees):
    if variant.get("n_type2_bees") is None and variant.get("n_type1_bees") == BCO_N_TYPE1_BEES:
        return max(1, n_bees // 2)
    return variant["n_type1_bees"]


def _variant_bco_cfg(city, spec, variant, *, weights=BASE_WEIGHTS, run_name_suffix=""):
    s = algo_settings(city)
    n_bees = int(s["bco_bees"])
    cfg = build_bco_cfg(
        run_name=f"{city}_{run_name_suffix}{variant['run_name']}",
        n_routes=spec["n_routes"], min_route_len=spec["min_route_len"],
        max_route_len=spec["max_route_len"], use_neural_bees=variant["use_neural_bees"],
        n_bees=n_bees, n_type1_bees=_variant_type1_bees(variant, n_bees),
        n_type2_bees=variant["n_type2_bees"], n_type4_bees=variant["n_type4_bees"],
        n_type5_bees=variant.get("n_type5_bees", 0),
        n_type6_bees=variant.get("n_type6_bees", 0),
        n_type7_bees=variant.get("n_type7_bees", 0),
        connectivity_mode=CONNECTIVITY_MODE,
        **s.get("bco_args", {}), **weights)
    bco_cfg_set(cfg, n_iterations=s["bco"])
    return cfg


# Two versions of our NBCO:
#   use_gnn=True  -> GNN construction bees (type-1) + our trim/extend edit bees (type-5)
#   use_gnn=False -> the GNN constructor is ALSO replaced by our trim/extend model
#                    (all bees type-5); isolates "drop the GNN, rely on our edit model".
# Two versions of our NBCO -- both = half rebuild bees (type-1) + half one-step
# edit bees (type-5). They differ only in WHICH model drives the rebuild bee:
#   use_gnn=True  -> rebuild bee = GNN construction model (standard NBCO)
#   use_gnn=False -> rebuild bee = OUR trim/edit model (rebuild via the edit
#                    model's RL rollout); type-5 is our edit model in both.
OUR_NBCO_VERSIONS = [(True, "GNN rebuild + trim/extend")]
if INCLUDE_EDIT_REBUILD:          # version B is ~250x slower/iter -> opt-in only
    OUR_NBCO_VERSIONS.append((False, "edit-model rebuild + trim/extend"))


def _our_model_cfg(city, spec, *, adj_target=None, run_name_suffix="", seed=None, use_gnn=True):
    s = algo_settings(city)
    n_bees = int(s["bco_bees"])
    rebuild_bees = max(1, n_bees // 2)         # rebuild bees (type-1)
    trim_extend_bees = n_bees - rebuild_bees   # one-step edit bees (type-5)
    if use_gnn:
        bee_arch, bee_w, tag = None, None, "our_nbco_gnn_rebuild_trimext"
    else:
        bee_arch = "bestsofar_feb2023_trim"    # rebuild driven by OUR edit model
        bee_w = str(OUR_MODEL_PATH)
        tag = "our_nbco_editmodel_rebuild_trimext"
    cfg = build_bco_cfg(
        run_name=f"{city}_{run_name_suffix}{tag}",
        n_routes=spec["n_routes"], min_route_len=spec["min_route_len"],
        max_route_len=spec["max_route_len"], use_neural_bees=True,
        n_bees=n_bees, n_type1_bees=rebuild_bees, n_type2_bees=0,
        n_type4_bees=0, n_type5_bees=trim_extend_bees,
        n_type6_bees=0, n_type7_bees=0,
        bee_model_arch=bee_arch, bee_model_weights=bee_w,
        connectivity_mode=CONNECTIVITY_MODE,
        **s.get("bco_args", {}), **OUR_WEIGHTS)
    bco_cfg_set(cfg, n_iterations=s["bco"])
    if adj_target is not None:
        bco_cfg_set(cfg, adjustment_degree_weight=float(ADJ_FIXED_W),
                    adjustment_degree_objective="cap",
                    adjustment_degree_target=float(adj_target),
                    adjustment_degree_gap=ADJ_GAP, adjustment_degree_mode=ADJ_MODE)
    if seed is not None:
        _set_cfg_value(cfg, "experiment.seed", int(seed))
    return cfg


def _run_our_model(city, spec, init, tensors, *, adj_target=None, run_name_suffix="",
                   seed=None, use_gnn=True):
    cfg = _our_model_cfg(city, spec, adj_target=adj_target,
                         run_name_suffix=run_name_suffix, seed=seed, use_gnn=use_gnn)
    return run_bco(cfg, init, tensors=tensors, run_name_scope=f"{city}_")


# --- experiment init = SAME pipeline as training: LC construction base seeded
#     with a MEDIUM redundancy tier (copy_mixed), NOT the clean LC and NOT NX.
#     lc_clean is computed separately, once, as a reference (cell below). ---
import random as _random
LC_INIT_WEIGHTS = dict(demand_time_weight=0.0, route_time_weight=0.5,
                       median_connectivity_weight=0.5)   # matches training objective
LC_INIT_SEED = 0
EXP_INIT_TIER = "covered_dup"   # tier injected into the experiment init network
EXP_TIER_CFG = {
    "copy_full":     {"events": 3, "max_multiplicity": 3, "kinds": ("full_copy",)},
    "copy_boundary": {"events": 4, "max_multiplicity": 4, "kinds": ("prefix_copy", "suffix_copy")},
    "copy_mixed":    {"events": 6, "max_multiplicity": 5,
                      "kinds": ("full_copy", "prefix_copy", "suffix_copy", "middle_copy")},
    # covered_dup: inject duplicates ONLY where coverage stays intact (d_un <= cap),
    # so the redundancy is purely removable (trim-only fixes it). Mirrors training.
    "covered_dup":   {"events": 6, "max_multiplicity": 4,
                      "kinds": ("full_copy", "prefix_copy", "suffix_copy", "middle_copy"),
                      "covered": True, "d_un_cap": 10.0},
}
_LC_INIT_CACHE = {}


# generic, contract-aware copy-redundancy injection (mirrors the training tiers)
def _tnodes(route):
    return [int(x) for x in route.tolist() if int(x) >= 0]


def _tsimple(ns, lo, hi):
    return (lo <= len(ns) <= hi and len(ns) == len(set(ns))
            and all(a != b for a, b in zip(ns[:-1], ns[1:])))


def _tredund(routes):
    cc = Counter()
    for r in routes:
        ns = _tnodes(r)
        for a, b in zip(ns[:-1], ns[1:]):
            cc[(min(a, b), max(a, b))] += 1
    t = sum(cc.values())
    return 0.0 if t == 0 else (t - len(cc)) / t


def _tfixed(routes, nr, hi):
    t = as_route_tensor(routes).long()
    if t.ndim == 3:
        t = t[0]
    if t.shape[0] < nr:
        t = torch.cat([t, torch.full((nr - t.shape[0], t.shape[1]), -1, dtype=t.dtype)], 0)
    else:
        t = t[:nr]
    if t.shape[1] < hi:
        t = torch.cat([t, torch.full((t.shape[0], hi - t.shape[1]), -1, dtype=t.dtype)], 1)
    elif t.shape[1] > hi:
        t = t[:, :hi]
    return t


def _tcand(routes, di, ti, kind, rng, lo, hi):
    donor, target = _tnodes(routes[di]), _tnodes(routes[ti])
    if not donor or not target:
        return None
    if kind == "full_copy":
        cand = donor
    else:
        ms = min(len(donor), len(target), 6)
        if kind == "middle_copy":
            ms = min(ms, len(target) - 2)
        if ms < 2:
            return None
        seg = rng.randint(2, ms)
        if kind == "prefix_copy":
            cand = donor[:seg] + target[seg:]
        elif kind == "suffix_copy":
            cand = target[:-seg] + donor[-seg:]
        elif kind == "middle_copy":
            ds = rng.randint(0, len(donor) - seg); ts = rng.randint(1, len(target) - seg - 1)
            cand = target[:ts] + donor[ds:ds + seg] + target[ts + seg:]
        else:
            raise ValueError(kind)
    if cand == target or not _tsimple(cand, lo, hi):
        return None
    return cand


def _trepl(routes, i, ns):
    routes[i] = -1
    routes[i, :len(ns)] = torch.as_tensor(ns, dtype=routes.dtype)


def _tuncovered(routes, demand, n):
    """Demand-weighted % of OD pairs not connected by any route (union-find)."""
    parent = list(range(n))
    def f(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for r in routes:
        ns = _tnodes(r)
        for a, b in zip(ns[:-1], ns[1:]):
            ra, rb = f(a), f(b)
            if ra != rb:
                parent[ra] = rb
    D = demand if torch.is_tensor(demand) else torch.as_tensor(demand)
    tot = unc = 0.0
    for a, b in (D > 0).nonzero(as_tuple=False).tolist():
        if a == b:
            continue
        w = float(D[a, b]); tot += w
        if f(a) != f(b):
            unc += w
    return 0.0 if tot == 0 else 100.0 * unc / tot


def _ttry(routes, kind, mm, rng, lo, hi, demand=None, n=None, cap=None, attempts=120):
    base = routes.clone()
    for _ in range(attempts):
        di = rng.randrange(routes.shape[0])
        tg = [i for i in range(routes.shape[0]) if i != di]; rng.shuffle(tg)
        want = rng.randint(1, min(mm - 1, len(tg))); mut = base.clone()
        cur = _tredund(mut); touched = 0
        for ti in tg:
            cand = _tcand(mut, di, ti, kind, rng, lo, hi)
            if cand is None:
                continue
            prop = mut.clone(); _trepl(prop, ti, cand)
            if _tredund(prop) <= cur + 1e-12:
                continue
            if cap is not None and _tuncovered(prop, demand, n) > cap:
                continue   # would break coverage -> reject (covered_dup tier)
            mut = prop; cur = _tredund(mut); touched += 1
            if touched >= want:
                return mut, touched
    return base, 0


def _inject_tier(routes, tier_cfg, rng, lo, hi, demand=None, n=None):
    routes = routes.clone()
    covered = bool(tier_cfg.get("covered", False))
    cap = float(tier_cfg.get("d_un_cap", 100.0)) if covered else None
    for _ in range(int(tier_cfg["events"])):
        ks = list(tier_cfg["kinds"]); rng.shuffle(ks)
        for k in ks:
            prop, t = _ttry(routes, k, int(tier_cfg["max_multiplicity"]), rng, lo, hi,
                            demand=demand, n=n, cap=cap)
            if t:
                routes = prop; break
    return routes


def _lc_base_routes(spec):
    """Clean LC construction routes for a city (no tier corruption)."""
    tensors = load_benchmark_tensors(spec["city"])
    cfg = build_lc_cfg(run_name=f"lc_init_{spec['city']}", n_routes=spec["n_routes"],
                       min_route_len=spec["min_route_len"],
                       max_route_len=spec["max_route_len"],
                       connectivity_mode=CONNECTIVITY_MODE, **LC_INIT_WEIGHTS)
    _set_cfg_value(cfg, "experiment.seed", int(LC_INIT_SEED))
    routes = run_lc(cfg, tensors=tensors, run_name_prefix=f"lc_init_{spec['city']}_", n_samples=1)[3]
    return tensors, _tfixed(routes, spec["n_routes"], spec["max_route_len"])


def load_benchmark_graph(spec, init_mode=None):     # overrides eval_lib.load_benchmark_graph
    city = spec["city"]
    if city not in _LC_INIT_CACHE:
        tensors, clean = _lc_base_routes(spec)
        rng = _random.Random(LC_INIT_SEED)
        init = _inject_tier(clean, EXP_TIER_CFG[EXP_INIT_TIER], rng,
                            spec["min_route_len"], spec["max_route_len"],
                            demand=tensors["demand"], n=tensors["node_locs"].shape[0])
        init = as_route_tensor(init)
        if init.ndim == 2:
            init = init[None]                      # -> (1, n_routes, max_len) for runners
        _LC_INIT_CACHE[city] = (tensors, init)
        print(f"[init] {city}: LC base + tier '{EXP_INIT_TIER}' -> "
              f"redun={redundancy_pct(_LC_INIT_CACHE[city][1]):.1f}%")
    return _LC_INIT_CACHE[city]


def _mean_over_seeds(rows):
    """Average numeric fields across per-seed row dicts (keep non-numeric / the
    first row's labels). Used to aggregate multi-seed BCO reruns."""
    base = dict(rows[0])
    for k, v in rows[0].items():
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            continue
        vals = [r[k] for r in rows
                if isinstance(r[k], (int, float)) and not isinstance(r[k], bool)
                and np.isfinite(r[k])]
        base[k] = float(np.mean(vals)) if vals else float("nan")
    return base


def adj_vs_init(routes, init_routes):
    r = as_route_tensor(routes); s = as_route_tensor(init_routes)
    if r.ndim == 2: r = r[None]
    if s.ndim == 2: s = s[None]
    nr = min(r.shape[1], s.shape[1]); w = min(r.shape[-1], s.shape[-1])
    return float(get_adjustment_degrees(
        r[:, :nr, :w], s[:, :nr, :w], True, gap=ADJ_GAP, mode=ADJ_MODE).mean().item())


def redundancy_pct(routes):
    R = as_route_tensor(routes)
    if R.ndim == 3: R = R[0]
    counts = Counter()
    for row in R:
        ns = [int(x) for x in row.tolist() if int(x) >= 0]
        for a, b in zip(ns[:-1], ns[1:]):
            counts[(min(a, b), max(a, b))] += 1
    t = sum(counts.values())
    return 0.0 if t == 0 else 100.0 * (t - len(counts)) / t


def conn_metric(m):
    v = metric_value(m, "median_connectivity_weighted")
    return v if np.isfinite(v) else metric_value(m, "median_connectivity")


def _full_metrics(m, rt, seed):
    """Full metric set for every results row: we optimize a subset (RTT+conn+adj)
    but always compute/report/save ALL of them."""
    rt = as_route_tensor(rt)
    return {
        "ATT": metric_value(m, "ATT"), "RTT": metric_value(m, "RTT"),
        "WMC": conn_metric(m), "WMC_mean": metric_value(m, "WMC_mean"),
        "adj_vs_seed": adj_vs_init(rt, seed),
        # normalized cost components -- the RTT / WMC the algorithm actually
        # optimizes (route_cost and median-connectivity, both /time_normalizer).
        "rtt_cost": metric_value(m, "cost_route_component"),
        "wmc_cost": metric_value(m, "cost_connectivity_component"),
        "cost": metric_value(m, "cost"),
        "d0": metric_value(m, "$d_0$"), "d1": metric_value(m, "$d_1$"),
        "d2": metric_value(m, "$d_2$"), "d_un": metric_value(m, "$d_{un}$"),
        "redun%": redundancy_pct(rt),
    }


# 2x2 ablation BCO variant (paper-style: GNN constructor swapped for RPC).
#   construct: "GNN" = neural route-construction rebuild (type-1, use_neural_bees)
#             "RPC" = random path-combiner rebuild (type-3, path_mix_rebuild)
#   edit:     "trim/extend" = our edit-model bee (type-5)
#             "type2"       = old local-endpoint edit (type-2)
ABL_MODELS = ["GNN + trim/extend", "RPC + trim/extend", "GNN + type2", "RPC + type2"]


def abl_variant(construct, edit, n_bees):
    half = max(1, n_bees // 2)
    edit_n = n_bees - half
    return {"run_name": f"abl_{construct}_{edit}".replace("/", ""),
            "summary_label": f"{construct} + {edit}",
            "use_neural_bees": construct == "GNN",
            "n_type1_bees": half if construct == "GNN" else 0,
            "n_type2_bees": edit_n if edit == "type2" else 0,
            "n_type4_bees": 0,
            "n_type5_bees": edit_n if edit == "trim/extend" else 0,
            "n_type6_bees": 0, "n_type7_bees": 0}


def nondominated_mask(df, xcol, ycol):
    pts = df[[xcol, ycol]].to_numpy(float)
    keep = []
    for x, y in pts:
        if not (np.isfinite(x) and np.isfinite(y)):
            keep.append(False); continue
        dom = (pts[:, 0] <= x) & (pts[:, 1] <= y) & ((pts[:, 0] < x) | (pts[:, 1] < y))
        keep.append(not dom.any())
    return np.array(keep)


# --- paper_results output sink: all NEW artifacts from this notebook land here ---
from eval_lib.results_io import ARTIFACTS_DIR
PAPER_DIR = ARTIFACTS_DIR / "paper_results"
PAPER_DIR.mkdir(parents=True, exist_ok=True)


def save_paper_table(df, name):
    path = PAPER_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"[paper] table ({len(df)} rows) -> {path}")
    return path


def reset_paper_table(name):
    path = PAPER_DIR / f"{name}.csv"
    if path.exists():
        path.unlink()
    return path


def append_paper_row(row, name, ndigits=3):
    path = PAPER_DIR / f"{name}.csv"
    header = not path.exists()
    pd.DataFrame([row]).round(ndigits).to_csv(path, mode="a", header=header, index=False)
    print(f"[paper] row -> {path}", flush=True)
    return path


def save_paper_fig(fig, name):
    # Figures are intentionally NOT persisted as images. The underlying data is
    # saved instead (CSV tables + route .pt dumps) so any figure can be rebuilt.
    print(f"[paper] figure '{name}' shown inline (rebuild from CSV / route dump)")
    return None


def save_paper_routes(name, routes, coords=None, street_adj=None, meta=None):
    """Dump a {label: route_tensor} mapping (+ coords/street_adj) to paper_results
    so the route figures can be reconstructed later. Mirrors the layout of
    artifacts/results/benchmark_<city>_routes.pt."""
    payload = {
        "routes": {k: as_route_tensor(v).cpu() for k, v in routes.items()},
        "coords": (coords.cpu() if hasattr(coords, "cpu") else coords),
        "street_adj": (street_adj.cpu() if hasattr(street_adj, "cpu") else street_adj),
        "meta": meta or {},
    }
    path = PAPER_DIR / f"{name}_routes.pt"
    torch.save(payload, path)
    print(f"[paper] route dump ({len(payload['routes'])} sets) -> {path}")
    return path


# --- shared run helpers (used by both the E1 legacy and E1u unified pipelines) ---
def _row(city, method, source, m, rt, seed, duration_s=None):
    return {"city": city, "method": method, "source": source,
            "duration_s": (round(float(duration_s), 1) if duration_s is not None else None),
            **_full_metrics(m, rt, seed)}


def _eval_routes_cfg(city, spec):
    """1-iteration SA cfg used only to evaluate a fixed route set's metrics."""
    return build_sa_cfg(f"{city}_eval", spec["n_routes"],
                        spec["min_route_len"], spec["max_route_len"],
                        n_iterations=1, connectivity_mode=CONNECTIVITY_MODE)


## E2 - Pareto fronts on RTT x WMC (Mumford0, covered_dup tier)

Both panels run on **Mumford0** from the **covered_dup** init tier (removable
redundancy), optimizing `alpha*RTT + (1-alpha)*WMC + adj(cap@target)` with
`use_weighted_connectivity=True`. `alpha` = `route_time_weight` (so alpha=1 ->
pure RTT, alpha=0 -> pure WMC).

- **Fig 1 (our model):** RTT x WMC Pareto front of `GNN + trim/extend`, swept
  over `alpha` x `adj_target` (one curve per target).
- **Fig 2 (4-model comparison):** `{GNN, RPC} x {trim/extend, type2}`, alpha-
  swept at a fixed `adj_target=OUR_MAIN_ADJ_TARGET` -> shows the trim/extend
  bee's contribution to the RTT x WMC front.

In [ ]:
# === E2 -- Pareto fronts on RTT x WMC (Mumford0, covered_dup init tier) ===
E2_CITY = "Mumford0" if not SMOKE else "Mandl"
E2_ALPHA_GRID = [0.0, 0.25, 0.5, 0.75, 1.0] if not SMOKE else [0.0, 0.5, 1.0]
E2_TARGET_GRID = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0] if not SMOKE else [0.0, 0.2]
E2_FIX_TARGET = min(E2_TARGET_GRID, key=lambda _t: abs(_t - 0.2))  # alpha-slice target
E2_FIX_ALPHA = min(E2_ALPHA_GRID, key=lambda _a: abs(_a - 0.5))    # target-slice alpha
E2_FIXED_TARGET = OUR_MAIN_ADJ_TARGET     # adj target for the 4-model comparison


def _e2_ctx():
    spec = next(s for s in BENCHMARK_SPECS if s["city"] == E2_CITY)
    tensors, init = load_benchmark_graph(spec)
    s = algo_settings(E2_CITY)
    return dict(city=E2_CITY, spec=spec, tensors=tensors, init=init,
                nb=int(s["bco_bees"]), settings=s)


def _run_rttwmc(ctx, construct, edit, *, alpha, adj_target, tag, seed=None):
    """One BCO run optimizing alpha*RTT + (1-alpha)*WMC + adj(cap@target)."""
    var = abl_variant(construct, edit, ctx["nb"])
    spec = ctx["spec"]
    weights = {"demand_time_weight": 0.0, "route_time_weight": float(alpha),
               "median_connectivity_weight": float(1.0 - alpha)}
    cfg = build_bco_cfg(
        run_name=f"{ctx['city']}_{tag}_{var['run_name']}",
        n_routes=spec["n_routes"], min_route_len=spec["min_route_len"],
        max_route_len=spec["max_route_len"], use_neural_bees=var["use_neural_bees"],
        n_bees=ctx["nb"], n_type1_bees=var["n_type1_bees"], n_type2_bees=var["n_type2_bees"],
        n_type4_bees=0, n_type5_bees=var["n_type5_bees"], n_type6_bees=0, n_type7_bees=0,
        connectivity_mode=CONNECTIVITY_MODE,
        **ctx["settings"].get("bco_args", {}), **weights)
    bco_cfg_set(cfg, n_iterations=int(E2_BCO_ITERATIONS))
    bco_cfg_set(cfg, adjustment_degree_weight=float(ADJ_FIXED_W),
                adjustment_degree_objective="cap", adjustment_degree_target=float(adj_target),
                adjustment_degree_gap=ADJ_GAP, adjustment_degree_mode=ADJ_MODE)
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
    if seed is not None:
        _set_cfg_value(cfg, "experiment.seed", int(seed))
    t = _time.perf_counter()
    res = run_bco(cfg, ctx["init"], tensors=ctx["tensors"], run_name_scope=f"{ctx['city']}_")
    duration_s = _time.perf_counter() - t
    return res[3], res[1], duration_s


def _e2_row(ctx, series_col, label, alpha, target, routes, metrics, duration_s=None):
    rt = as_route_tensor(routes)
    return {"city": ctx["city"], series_col: label, "alpha": float(alpha),
            "adj_target": float(target),
            "duration_s": (round(float(duration_s), 1) if duration_s is not None else None),
            **_full_metrics(metrics, rt, ctx["init"])}


_AXIS_LABEL = {"RTT": "RTT (lower = better)", "WMC": "WMC (lower = better)",
              "adj_vs_seed": "adjustment degree vs seed (lower = less change)"}


def _plot_fronts(df, series_col, title, save_name, xcol="RTT", ycol="WMC"):
    fig, ax = plt.subplots(figsize=(8, 6))
    for lab, sub in df.groupby(series_col):
        sub = sub.sort_values(xcol)
        ax.plot(sub[xcol], sub[ycol], marker="o", alpha=0.85, label=str(lab))
        for _, r in sub.iterrows():
            ax.annotate(f"a={r['alpha']:g},t={r['adj_target']:g}", (r[xcol], r[ycol]),
                        fontsize=6, alpha=0.7, xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel(_AXIS_LABEL.get(xcol, xcol)); ax.set_ylabel(_AXIS_LABEL.get(ycol, ycol))
    ax.set_title(title); ax.legend(title=series_col); ax.grid(alpha=0.3)
    save_paper_fig(fig, save_name); plt.show(); plt.close(fig)


_ctx = _e2_ctx()
_sfx = "_smoke" if SMOKE else ""
print(f"=== E2 Pareto fronts on RTT x WMC ({E2_CITY}, tier={EXP_INIT_TIER}) ===")

# --- Fig 1: OUR model (GNN + trim/extend), Pareto over alpha x target ---
_e2_our_table = f"final_e2_ourpareto_{E2_CITY}{_sfx}"
reset_paper_table(_e2_our_table)
_our_rows = []
for _t in tqdm(E2_TARGET_GRID, desc=f"E2 {E2_CITY} our-model targets"):
    for _a in E2_ALPHA_GRID:
        try:
            r, m, dt = _run_rttwmc(_ctx, "GNN", "trim/extend", alpha=_a, adj_target=_t,
                               tag=f"our_a{_a}_t{_t}", seed=SEEDS[0])
            row = _e2_row(_ctx, "adj_target", f"target={_t:g}", _a, _t, r, m, duration_s=dt)
            _our_rows.append(row)
            append_paper_row(row, _e2_our_table, ndigits=4)
        except Exception as exc:
            print(f"  our alpha={_a} target={_t} FAILED: {exc}")
E2_OUR_DF = pd.DataFrame(_our_rows)
if not E2_OUR_DF.empty:
    display(E2_OUR_DF.round(4))
    save_paper_table(E2_OUR_DF.round(4), _e2_our_table)
    # (1) combined: one curve per adj_target, each curve = alpha sweep
    _plot_fronts(E2_OUR_DF, "adj_target",
                 f"E2 ({E2_CITY}): our model RTT x WMC -- alpha x target (combined)",
                 f"final_e2_ourpareto_combined_{E2_CITY}{_sfx}")
    # (1b) SAME data on adj x RTT axes -- makes the target sweep (which moves the
    # deviation, not RTT/WMC directly) visible: points spread along the adj axis.
    _plot_fronts(E2_OUR_DF, "adj_target",
                 f"E2 ({E2_CITY}): our model adj x RTT -- alpha x target",
                 f"final_e2_ourpareto_adjrtt_{E2_CITY}{_sfx}",
                 xcol="adj_vs_seed", ycol="RTT")
    # (2) alpha sweep at a fixed adj target
    _alpha_slice = E2_OUR_DF[np.isclose(E2_OUR_DF["adj_target"], E2_FIX_TARGET)]
    if not _alpha_slice.empty:
        _plot_fronts(_alpha_slice, "adj_target",
                     f"E2 ({E2_CITY}): alpha sweep on RTT x WMC (target={E2_FIX_TARGET:g} fixed)",
                     f"final_e2_alphasweep_{E2_CITY}{_sfx}")
    # (3) adj-target sweep at a fixed alpha
    _target_slice = E2_OUR_DF[np.isclose(E2_OUR_DF["alpha"], E2_FIX_ALPHA)]
    if not _target_slice.empty:
        _plot_fronts(_target_slice, "alpha",
                     f"E2 ({E2_CITY}): adj-target sweep on RTT x WMC (alpha={E2_FIX_ALPHA:g} fixed)",
                     f"final_e2_targetsweep_{E2_CITY}{_sfx}")

# --- Fig 2: 4-model comparison, alpha-swept at fixed adj target ---
_e2_abl_table = f"final_e2_4model_{E2_CITY}{_sfx}"
reset_paper_table(_e2_abl_table)
_abl_rows = []
for _mdl in tqdm(ABL_MODELS, desc=f"E2 {E2_CITY} 4-model alpha sweep"):
    _c, _e = _mdl.split(" + ")
    for _a in E2_ALPHA_GRID:
        try:
            r, m, dt = _run_rttwmc(_ctx, _c, _e, alpha=_a, adj_target=E2_FIXED_TARGET,
                               tag=f"abl_a{_a}".replace(".", ""), seed=SEEDS[0])
            row = _e2_row(_ctx, "model", _mdl, _a, E2_FIXED_TARGET, r, m, duration_s=dt)
            _abl_rows.append(row)
            append_paper_row(row, _e2_abl_table, ndigits=4)
        except Exception as exc:
            print(f"  {_mdl} alpha={_a} FAILED: {exc}")
E2_ABL_DF = pd.DataFrame(_abl_rows)
if not E2_ABL_DF.empty:
    display(E2_ABL_DF.round(4))
    save_paper_table(E2_ABL_DF.round(4), _e2_abl_table)
    _plot_fronts(E2_ABL_DF, "model",
                 f"E2 ({E2_CITY}): 4-model RTT x WMC fronts (alpha sweep @ target={E2_FIXED_TARGET:g})",
                 f"final_e2_4model_{E2_CITY}{_sfx}")
print(f"[E2] {E2_CITY}: our-model alpha x target front + 4-model alpha sweep done.")


[init] Mumford0: LC base + tier 'covered_dup' -> redun=31.7%
=== E2 Pareto fronts on RTT x WMC (Mumford0, tier=covered_dup) ===


E2 Mumford0 our-model targets:   0%|          | 0/6 [00:00<?, ?it/s]

[Mumford0_our_a0.0_t0.0_abl_GNN_trimextend] route_selection=uniform-random | bee split: neural_rebuild=5 local_endpoint_edit=0 path_mix_rebuild=0 construction_extend=0 extend_trim_edit=5 trim_only=0 trim_then_extend=0 (total=10)


100%|██████████| 120/120 [04:34<00:00,  2.29s/it]


,12.000,1.154,27.334,392.000,19.228,35.191,30.772,14.809,0.000,0.000,20.233,20.833,274.546,121.000
[Mumford0_our_a0.25_t0.0_abl_GNN_trimextend] route_selection=uniform-random | bee split: neural_rebuild=5 local_endpoint_edit=0 path_mix_rebuild=0 construction_extend=0 extend_trim_edit=5 trim_only=0 trim_then_extend=0 (total=10)


...

,12.000,0.895,24.228,268.000,17.816,39.251,33.841,9.092,0.000,0.000,17.633,18.233,387.291,121.000
[Mumford0_our_a0.25_t0.2_abl_GNN_trimextend] route_selection=uniform-random | bee split: neural_rebuild=5 local_endpoint_edit=0 path_mix_rebuild=0 construction_extend=0 extend_trim_edit=5 trim_only=0 trim_then_extend=0 (total=10)


100%|██████████| 120/120 [04:39<00:00,  2.33s/it]


,12.000,0.808,30.493,252.000,15.338,25.845,27.069,31.748,0.000,0.000,21.767,21.900,279.129,121.000
[Mumford0_our_a0.5_t0.2_abl_GNN_trimextend] route_selection=uniform-random | bee split: neural_rebuild=5 local_endpoint_edit=0 path_mix_rebuild=0 construction_extend=0 extend_trim_edit=5 trim_only=0 trim_then_extend=0 (total=10)


100%|██████████| 120/120 [04:37<00:00,  2.32s/it]


,12.000,0.808,30.493,252.000,15.338,25.845,27.069,31.748,0.000,0.000,21.767,21.900,278.050,121.000
[Mumford0_our_a0.75_t0.2_abl_GNN_trimextend] route_selection=uniform-random | bee split: neural_rebuild=5 local_endpoint_edit=0 path_mix_rebuild=0 construction_extend=0 extend_trim_edit=5 trim_only=0 trim_then_extend=0 (total=10)


## E1 - Main cross-city comparison (legacy ATT/RTT baselines, from saved files)

These are the legacy ATT+RTT (alpha=0.5) baselines. They are **precomputed**: nothing
runs live unless a city is listed in `E1_RUN_CITIES` (empty by default). The combined
table is assembled from the saved `paper_results/final_main_<city>.csv` dumps. The
apples-to-apples unified comparison is **E1u** below; the Pareto ablation is **E2** above.

In [ ]:
# === E1 -- legacy ATT+RTT baselines (alpha=0.5, connectivity off) ===
# Every method optimizes BASE_WEIGHTS (demand 0.5 + route 0.5, connectivity 0).
# Precomputed: loaded from saved CSVs unless a city is added to E1_RUN_CITIES.
# Our model lives in the unified E1u table below (it optimizes RTT+WMC+adj).

def _legacy_weights(cfg):
    """Set a baseline cfg to the legacy ATT+RTT objective (demand/route .5, conn 0)."""
    for _k in ("demand_time_weight", "route_time_weight", "median_connectivity_weight"):
        _set_cfg_value(cfg, f"experiment.cost_function.kwargs.{_k}", BASE_WEIGHTS[_k])
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.connectivity_mode", CONNECTIVITY_MODE)
    return cfg


def run_city(spec, table_name=None, comparison_table_name=None):
    city = spec["city"]
    nr, mn, mx = spec["n_routes"], spec["min_route_len"], spec["max_route_len"]
    tensors, init = load_benchmark_graph(spec)
    seed = as_route_tensor(init)
    rows, routes = [], {}

    def add(method, m, rt, dt=None):
        rt = as_route_tensor(rt)
        row = _row(city, method, "rerun", m, rt, seed, duration_s=dt)
        rows.append(row); routes[method] = rt
        if table_name is not None:
            append_paper_row(row, table_name, ndigits=3)
        if comparison_table_name is not None:
            append_paper_row(row, comparison_table_name, ndigits=3)
        tail = f"  ({dt:.0f}s)" if dt is not None else ""
        print(f"  [done] {method:36} ATT={row['ATT']:.2f} RTT={row['RTT']:.0f} "
              f"WMC={row['WMC']:.2f} cost={row['cost']:.3f} adj={row['adj_vs_seed']:.2f} "
              f"redun={row['redun%']:.0f}%{tail}", flush=True)

    def run_one(name, fn):
        print(f"  -> {name} ...", flush=True)
        t = _time.perf_counter()
        try:
            return fn(), _time.perf_counter() - t
        except Exception as exc:
            print(f"  {city}/{name} FAILED: {exc}", flush=True)
            return None, None

    r, dt = run_one(f"Initial (LC+{EXP_INIT_TIER})", lambda: _run_baseline(
        None, _legacy_weights(_eval_routes_cfg(city, spec)), init,
        f"{city}_init_", {}, tensors=tensors))
    if r is not None:
        add(f"Initial (LC+{EXP_INIT_TIER})", r[1], r[3], dt)

    for name, builder in [
        ("Simulated annealing", lambda: run_sa(
            _legacy_weights(_sa_cfg(city, f"{city}_sa", nr, mn, mx)), init,
            tensors=tensors, run_name_scope=f"{city}_")),
        ("Genetic algorithm", lambda: run_ga(
            _legacy_weights(_ga_cfg(city, f"{city}_ga", nr, mn, mx)), init,
            tensors=tensors, run_name_scope=f"{city}_")),
        ("Hyper-heuristics", lambda: run_hh(
            _legacy_weights(_hh_cfg(city, f"{city}_hh", nr, mn, mx)), init,
            tensors=tensors, run_name_scope=f"{city}_")),
    ]:
        r, dt = run_one(name, builder)
        if r is not None:
            add(name, r[1], r[3], dt)

    for label, vkey in [("BCO", "seeded_heuristic"), ("neural BCO", "seeded_neural")]:
        v = next(x for x in BCO_VARIANTS if x["key"] == vkey)
        # _variant_bco_cfg defaults to weights=BASE_WEIGHTS -> same ATT+RTT objective.
        r, dt = run_one(label, lambda v=v: run_bco(
            _variant_bco_cfg(city, spec, v), init, tensors=tensors,
            run_name_scope=f"{city}_"))
        if r is not None:
            add(label, r[1], r[3], dt)

    def _nsga():
        it, pop = algo_settings(city)["nsgaii"]
        _, out = run_nsgaii(build_nsgaii_cfg(f"{city}_nsgaii", nr, mn, mx,
                            n_iterations=it, pop_size=pop,
                            connectivity_mode=CONNECTIVITY_MODE),
                            tensors=tensors, init_routes=init, run_name_scope=f"{city}_",
                            connectivity_mode=CONNECTIVITY_MODE)
        best = reduce_pareto_front(out, BASE_WEIGHTS["demand_time_weight"],
                                   BASE_WEIGHTS["route_time_weight"])
        rr = best["routes"]; rr = rr[None] if rr.ndim == 2 else rr
        return _run_baseline(None, _legacy_weights(_eval_routes_cfg(city, spec)), rr,
                             f"{city}_nsgaii_eval_", {}, tensors=tensors)
    r, dt = run_one("NSGA-II", _nsga)
    if r is not None:
        add("NSGA-II", r[1], r[3], dt)

    return rows, routes, tensors, init


all_rows = []
_sfx = "_smoke" if SMOKE else ""
_e1_comparison_table = "final_main_comparison" + _sfx
if E1_RUN_CITIES:
    reset_paper_table(_e1_comparison_table)
for spec in [s for s in BENCHMARK_SPECS if s["city"] in E1_RUN_CITIES]:
    print(f"=== {spec['city']} (legacy ATT+RTT baselines, single seed) ===", flush=True)
    _city_table = f"final_main_{spec['city']}{_sfx}"
    reset_paper_table(_city_table)
    rows, routes, tensors, init = run_city(
        spec, table_name=_city_table, comparison_table_name=_e1_comparison_table)
    all_rows += rows
    save_paper_table(pd.DataFrame(rows).round(3), _city_table)
    save_paper_routes(_city_table, routes,
                      tensors["node_locs"], tensors["street_adj"], meta={"city": spec["city"]})
    print(f"=== {spec['city']} done ({len(rows)} methods) ===", flush=True)

# Combined E1 table = saved per-city CSVs (every city is precomputed; only
# E1_RUN_CITIES, if any, are re-run live above and overwrite their CSV).
_e1_frames = []
for _city in CITIES:
    _csv = PAPER_DIR / f"final_main_{_city}{_sfx}.csv"
    if _csv.exists():
        _e1_frames.append(pd.read_csv(_csv))
    else:
        print(f"[E1] no saved table for {_city} ({_csv.name}) -- add it to E1_RUN_CITIES")
main_df = (pd.concat(_e1_frames, ignore_index=True).round(3) if _e1_frames
           else pd.DataFrame(all_rows).round(3))
display(main_df)
save_paper_table(main_df, _e1_comparison_table)
print(f"E1 combined from {len(_e1_frames)} saved city tables "
      f"(live re-runs this pass: {E1_RUN_CITIES or 'none'}).")

[E1] no saved table for Mumford3 (final_main_Mumford3.csv) -- add it to E1_RUN_CITIES


,city,method,source,ATT,RTT,WMC,adj_vs_seed,cost,d0,d1,d2,d_un,redun%
0,Mandl,Initial (LC+covered_dup),rerun,18.150,366.0,2.535,0.000,2.517,56.904,35.324,0.000,7.771,64.516
1,Mandl,Simulated annealing,rerun,13.420,140.0,1.616,0.651,0.384,65.832,27.039,6.423,0.706,6.250
2,Mandl,Genetic algorithm,rerun,13.712,136.0,1.617,0.808,0.380,60.951,28.388,8.863,1.798,0.000
3,Mandl,Hyper-heuristics,rerun,15.803,138.0,1.652,0.727,0.405,51.445,39.949,8.092,0.514,6.667
4,Mandl,BCO,rerun,13.645,136.0,1.624,0.476,0.550,56.776,35.710,6.487,1.028,0.000
5,Mandl,neural BCO,rerun,13.496,130.0,1.640,0.532,0.533,56.712,37.636,5.267,0.385,0.000
6,Mandl,NSGA-II,rerun,13.968,138.0,1.662,0.715,0.386,57.547,38.664,3.789,0.000,6.667
7,Mandl,Our NBCO (GNN rebuild + trim/extend),rerun,18.417,208.0,1.892,0.202,1.051,58.253,41.426,0.321,0.000,26.316
8,Mumford0,Initial (LC+covered_dup),rerun,28.232,372.0,8.652,0.000,1.715,16.256,29.989,28.180,25.576,31.707
9,Mumford0,Simulated annealing,rerun,21.353,430.0,7.281,0.860,0.818,29.755,45.359,22.872,2.014,22.000


[paper] table (32 rows) -> D:\PythonProjects\connectpt\artifacts\paper_results\final_main_comparison.csv
E1 combined from 4 saved city tables (live re-runs this pass: none).


## E1u - Unified objective: ALL methods optimize 0.5*RTT + 0.5*WMC + adj

Every method (SA / GA / HH / NSGA-II / BCO / neural BCO / our model) runs on the
**same** unified objective: route time + weighted mean connectivity + the
adjustment-degree penalty (cap @ `OUR_MAIN_ADJ_TARGET`, weight `ADJ_FIXED_W`).
This is the apples-to-apples comparison; E1 above is the legacy ATT/RTT baseline.
Cities: `E1U_CITIES` (Mandl + Mumford0 + Mumford1). Tables ->
`paper_results/final_main_unified_<city>.csv`.

In [ ]:
import torch

UNIFIED_ADJ = dict(adjustment_degree_weight=float(ADJ_FIXED_W),
                   adjustment_degree_target=float(OUR_MAIN_ADJ_TARGET),
                   adjustment_degree_objective="cap",
                   adjustment_degree_gap=ADJ_GAP, adjustment_degree_mode=ADJ_MODE)


def _unify_weights(cfg):
    """Set a baseline cfg's cost weights to the unified objective (RTT+WMC, demand off)."""
    for _k, _v in (("demand_time_weight", 0.0), ("route_time_weight", 0.5),
                   ("median_connectivity_weight", 0.5)):
        _set_cfg_value(cfg, f"experiment.cost_function.kwargs.{_k}", _v)
    _set_cfg_value(cfg, "experiment.cost_function.kwargs.connectivity_mode", CONNECTIVITY_MODE)
    return cfg


def _ravel_hist(h):
    """Flatten a per-sample cost history to a plain list of floats (or [])."""
    if h is None:
        return []
    arr = np.asarray(h.detach().cpu() if hasattr(h, "detach") else h).ravel()
    return [float(v) for v in arr]


def run_city_unified(spec, table_name=None, comparison_table_name=None):
    city = spec["city"]
    nr, mn, mx = spec["n_routes"], spec["min_route_len"], spec["max_route_len"]
    tensors, init = load_benchmark_graph(spec)
    seed = as_route_tensor(init)
    rows, routes, conv = [], {}, {}

    def add(method, m, rt, dt=None, hist=None):
        rt = as_route_tensor(rt)
        row = _row(city, method, "unified", m, rt, seed, duration_s=dt)
        rows.append(row); routes[method] = rt
        if table_name is not None:
            append_paper_row(row, table_name, ndigits=3)
        if comparison_table_name is not None:
            append_paper_row(row, comparison_table_name, ndigits=3)
        if hist is not None:
            conv[method] = _ravel_hist(hist)
        tail = f"  ({dt:.0f}s)" if dt is not None else ""
        print(f"  [done] {method:34} RTT={row['RTT']:.0f} WMC={row['WMC']:.2f} "
              f"adj={row['adj_vs_seed']:.2f} cost={row['cost']:.3f} "
              f"redun={row['redun%']:.0f}%{tail}", flush=True)

    def run_one(name, fn):
        print(f"  -> {name} ...", flush=True)
        t = _time.perf_counter()
        try:
            return fn(), _time.perf_counter() - t
        except Exception as exc:
            print(f"  {city}/{name} FAILED: {exc}", flush=True); return None, None

    r, dt = run_one(f"Initial (LC+{EXP_INIT_TIER})", lambda: _run_baseline(
        None, _unify_weights(_eval_routes_cfg(city, spec)), init,
        f"{city}_uini_", {}, tensors=tensors, use_weighted_connectivity=True))
    if r is not None:
        add(f"Initial (LC+{EXP_INIT_TIER})", r[1], r[3], dt)

    for name, builder in [
        ("Simulated annealing", lambda: run_sa(
            _unify_weights(_sa_cfg(city, f"{city}_usa", nr, mn, mx)), init, tensors=tensors,
            run_name_scope=f"{city}_u", use_weighted_connectivity=True,
            return_history=True, **UNIFIED_ADJ)),
        ("Genetic algorithm", lambda: run_ga(
            _unify_weights(_ga_cfg(city, f"{city}_uga", nr, mn, mx)), init, tensors=tensors,
            run_name_scope=f"{city}_u", use_weighted_connectivity=True,
            return_history=True, **UNIFIED_ADJ)),
        ("Hyper-heuristics", lambda: run_hh(
            _unify_weights(_hh_cfg(city, f"{city}_uhh", nr, mn, mx)), init, tensors=tensors,
            run_name_scope=f"{city}_u", use_weighted_connectivity=True,
            return_history=True, **UNIFIED_ADJ)),
    ]:
        r, dt = run_one(name, builder)
        if r is not None:
            _h = r[-1]
            add(name, r[1], r[3], dt,
                hist=(_h[0] if (_h is not None and len(_h) > 0) else None))

    for label, vkey in [("BCO", "seeded_heuristic"), ("neural BCO", "seeded_neural")]:
        v = next(x for x in BCO_VARIANTS if x["key"] == vkey)
        ch = {}

        def _bco(v=v, ch=ch):
            cfg = _variant_bco_cfg(city, spec, v, weights=OUR_WEIGHTS, run_name_suffix="uni_")
            bco_cfg_set(cfg, adjustment_degree_weight=float(ADJ_FIXED_W),
                        adjustment_degree_objective="cap",
                        adjustment_degree_target=float(OUR_MAIN_ADJ_TARGET),
                        adjustment_degree_gap=ADJ_GAP, adjustment_degree_mode=ADJ_MODE)
            _set_cfg_value(cfg, "experiment.cost_function.kwargs.use_weighted_connectivity", True)
            return run_bco(cfg, init, tensors=tensors, run_name_scope=f"{city}_",
                           cost_history_out=ch)
        r, dt = run_one(label, _bco)
        if r is not None:
            add(label, r[1], r[3], dt, hist=ch.get("history"))

    def _nsga():
        it, pop = algo_settings(city)["nsgaii"]
        _, out = run_nsgaii(build_nsgaii_cfg(f"{city}_unsgaii", nr, mn, mx,
                            n_iterations=it, pop_size=pop,
                            connectivity_mode=CONNECTIVITY_MODE),
                            tensors=tensors, init_routes=init, run_name_scope=f"{city}_u",
                            use_weighted_connectivity=True,
                            connectivity_mode=CONNECTIVITY_MODE, **UNIFIED_ADJ)
        best = reduce_pareto_front(out, 0.5, 0.5)   # objectives are (RTT, WMC)
        rr = best["routes"]; rr = rr[None] if rr.ndim == 2 else rr
        return _run_baseline(None, _unify_weights(_eval_routes_cfg(city, spec)), rr,
                             f"{city}_unsga_eval_", {}, tensors=tensors,
                             use_weighted_connectivity=True,
                             adjustment_seed_routes=init, **UNIFIED_ADJ)
    r, dt = run_one("NSGA-II", _nsga)
    if r is not None:
        add("NSGA-II", r[1], r[3], dt)

    # our model already optimizes the unified objective (RTT + WMC + adj cap)
    for _ug, _lbl in OUR_NBCO_VERSIONS:
        method = f"Our NBCO ({_lbl})"
        ch = {}
        r, dt = run_one(method, lambda ug=_ug, ch=ch: run_bco(
            _our_model_cfg(city, spec, adj_target=OUR_MAIN_ADJ_TARGET, seed=SEEDS[0], use_gnn=ug),
            init, tensors=tensors, run_name_scope=f"{city}_", cost_history_out=ch))
        if r is not None:
            add(method, r[1], r[3], dt, hist=ch.get("history"))
    return rows, routes, tensors, init, conv


u_all_rows = []
_sfx = "_smoke" if SMOKE else ""
_e1u_comparison_table = "final_main_unified_comparison" + _sfx
if E1U_CITIES:
    reset_paper_table(_e1u_comparison_table)
for spec in [s for s in BENCHMARK_SPECS if s["city"] in E1U_CITIES]:
    print(f"=== {spec['city']} UNIFIED (all methods on 0.5*RTT+0.5*WMC+adj) ===", flush=True)
    _city_table = f"final_main_unified_{spec['city']}{_sfx}"
    reset_paper_table(_city_table)
    rows, routes, tensors, init, conv = run_city_unified(
        spec, table_name=_city_table, comparison_table_name=_e1u_comparison_table)
    u_all_rows += rows
    save_paper_table(pd.DataFrame(rows).round(3), _city_table)
    save_paper_routes(_city_table, routes,
                      tensors["node_locs"], tensors["street_adj"],
                      meta={"city": spec["city"], "objective": "unified"})
    torch.save(conv, PAPER_DIR / f"final_main_unified_{spec['city']}{_sfx}_history.pt")
    print(f"[paper] cost histories ({len(conv)} methods) -> "
          f"final_main_unified_{spec['city']}{_sfx}_history.pt", flush=True)
    print(f"=== {spec['city']} unified done ({len(rows)} methods) ===", flush=True)

unified_df = pd.DataFrame(u_all_rows).round(3)
display(unified_df)
save_paper_table(unified_df, _e1u_comparison_table)
print("E1u: every method optimized 0.5*RTT + 0.5*WMC + adj(cap@%.2f); "
      "tables/route dumps/histories -> paper_results/final_main_unified_*." % OUR_MAIN_ADJ_TARGET)


=== Mandl UNIFIED (all methods on 0.5*RTT+0.5*WMC+adj) ===
[init] Mandl: LC base + tier 'covered_dup' -> redun=64.5%
,6.000,2.625,18.150,366.000,56.904,35.324,0.000,7.771,20.000,0.000,18.167,2.535,0.012,0.000
  [done] Initial (LC+covered_dup)           RTT=366 WMC=2.53 adj=0.00 cost=2.625 redun=65%
  -> Simulated annealing ...


100%|██████████| 15000/15000 [03:22<00:00, 74.18it/s]

,6.000,0.504,22.490,188.000,41.618,45.922,11.496,0.963,0.000,0.000,20.067,1.898,202.224,15001.000
  [done] Simulated annealing                RTT=188 WMC=1.90 adj=0.20 cost=0.504 redun=12%  (202s)
  -> Genetic algorithm ...



100%|██████████| 200/200 [00:13<00:00, 14.93it/s]

,6.000,0.539,18.318,202.000,60.758,38.407,0.835,0.000,0.000,0.000,18.200,1.892,13.503,201.000
  [done] Genetic algorithm                  RTT=202 WMC=1.89 adj=0.18 cost=0.539 redun=22%  (14s)
  -> Hyper-heuristics ...


...

,15.000,2.133,32.083,1408.000,21.222,41.523,27.133,10.123,0.000,0.000,25.414,11.178,633.800,251.000
  [done] BCO                                RTT=1408 WMC=11.18 adj=0.20 cost=2.133 redun=25%  (634s)
  -> neural BCO ...


[Mumford1_uni_seeded_neural_bco_from_lc_mumford0] route_selection=uniform-random | bee split: neural_rebuild=5 local_endpoint_edit=5 path_mix_rebuild=0 construction_extend=0 extend_trim_edit=0 trim_only=0 trim_then_extend=0 (total=10)


100%|██████████| 250/250 [1:38:35<00:00, 23.66s/it]

,15.000,2.082,35.989,1374.000,18.934,40.701,31.002,9.363,0.000,0.000,29.986,12.620,5915.550,251.000
  [done] neural BCO                         RTT=1374 WMC=12.62 adj=0.21 cost=2.082 redun=34%  (5916s)
  -> NSGA-II ...



100%|██████████| 400/400 [00:07<00:00, 53.43it/s]
100%|██████████| 60.00/60[03:13<00:00]
 76%|███████▌  | 604/800 [22:56<06:22,  1.95s/it]

### Route visualisation (unified run: RTT + WMC + adj for all methods)

Drawn from the **unified** dumps (`final_main_unified_<city>`). Layout: OD demand -> init -> methods (plain & diff vs init), plus a focused 1x4 (our routes / init / neural-BCO diff / our-NBCO diff).

In [ ]:
# Autonomous route-set visualisation (reads paper_results dumps). Layout:
#   grid 1 (plain): [OD demand] [init] [methods...]
#   grid 2 (diff):  [OD demand] [init] [methods diff vs init]
#   + focused 1x4: our routes / init / neural-BCO diff / our-NBCO diff.
import math
import torch
import pandas as pd
import matplotlib.pyplot as plt
from eval_lib.results_io import ARTIFACTS_DIR
from eval_lib import plots as route_plots
from eval_lib import as_route_tensor, load_benchmark_tensors

VIZ_CITY = "Mumford0"          #  UNIFIED route dumps
VIZ_OVERLAP_CURVES = True      # overlap "ribbons"; set False for dense Mumford networks
VIZ_NCOL = 3
VIZ_DEMAND_TOP_FRAC = None     # None = all OD pairs; e.g. 0.15 keeps only busiest 15%

_prdir = ARTIFACTS_DIR / "paper_results"
_vsfx = "_smoke" if SMOKE else ""
_ptf = _prdir / f"final_main_unified_{VIZ_CITY}{_vsfx}_routes.pt"
assert _ptf.exists(), f"no route dump: {_ptf} (run E1u for {VIZ_CITY} first)"
_dump = torch.load(_ptf, weights_only=False)
_routes, _coords, _adj = _dump["routes"], _dump["coords"], _dump["street_adj"]
_methods = list(_routes)
_demand = load_benchmark_tensors(VIZ_CITY)["demand"]   # OD matrix for the demand panel

# initial network = reference for the diff version
_ref_key = next((m for m in _methods if "Initial" in m), _methods[0])
_ref_rt = as_route_tensor(_routes[_ref_key]); _ref_rr = _ref_rt[0] if _ref_rt.ndim == 3 else _ref_rt

# optional metric subtitle from the matching results table
_sub = {}
_csvf = _prdir / f"final_main_unified_{VIZ_CITY}{_vsfx}.csv"
if _csvf.exists():
    _df = pd.read_csv(_csvf).set_index("method")
    for _m, _r in _df.iterrows():
        _sub[_m] = (f"RTT={_r['RTT']:.0f}  WMC={_r['WMC']:.2f}  "
                    f"adj={_r['adj_vs_seed']:.2f}  redun={_r['redun%']:.0f}%")


def _draw_panel(ax, key, diff):
    if key == "__demand__":
        route_plots.plot_demand_graph(ax, _demand, _coords, _adj, title="OD demand",
                                      subtitle="edge color/width = demand volume",
                                      top_frac=VIZ_DEMAND_TOP_FRAC)
        return
    _rt = as_route_tensor(_routes[key]); _rr = _rt[0] if _rt.ndim == 3 else _rt
    try:
        if diff and key != _ref_key:
            route_plots.plot_route_diff(ax, _rr, _ref_rr, _coords, _adj, title=key,
                                        subtitle=_sub.get(key, ""), palette="tab20",
                                        with_overlap_curves=VIZ_OVERLAP_CURVES)
        else:
            route_plots.plot_plain_route_set(ax, _rr, _coords, _adj, title=key,
                                             subtitle=_sub.get(key, ""), palette="tab20",
                                             with_overlap_curves=VIZ_OVERLAP_CURVES)
    except Exception as exc:
        ax.set_title(f"{key}: plot failed ({exc})")


def _draw_grid(diff):
    # panel order: demand first, init second, then every other method
    panels = ["__demand__", _ref_key] + [m for m in _methods if m != _ref_key]
    _nrow = math.ceil(len(panels) / VIZ_NCOL)
    fig, axes = plt.subplots(_nrow, VIZ_NCOL, figsize=(6.5 * VIZ_NCOL, 6.5 * _nrow), squeeze=False)
    for _ax, _panel in zip(axes.flat, panels):
        _draw_panel(_ax, _panel, diff)
    for _ax in axes.flat[len(panels):]:
        _ax.axis("off")
    _kind = f"DIFF vs {_ref_key}" if diff else "plain route sets"
    fig.suptitle(f"{VIZ_CITY} UNIFIED: demand + init + methods ({_kind})",
                 fontsize=15, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show(); plt.close(fig)


_draw_grid(diff=False)   # version 1: plain route sets
_draw_grid(diff=True)    # version 2: diff vs the initial network


# --- focused 1x4: our routes / init / neural-BCO diff / our-NBCO diff ---
_our_key = next((m for m in _methods if "Our NBCO" in m), None)
_nbco_key = next((m for m in _methods if m == "neural BCO"),
                 next((m for m in _methods if "neural" in m.lower()), None))
if _our_key and _nbco_key:
    _ourt = as_route_tensor(_routes[_our_key]); _our_rr = _ourt[0] if _ourt.ndim == 3 else _ourt
    _nbt = as_route_tensor(_routes[_nbco_key]); _nb_rr = _nbt[0] if _nbt.ndim == 3 else _nbt
    fig, axes = plt.subplots(1, 4, figsize=(26, 6.5), squeeze=False)
    route_plots.plot_plain_route_set(axes[0, 0], _our_rr, _coords, _adj,
                                     title="Our NBCO -- routes", subtitle=_sub.get(_our_key, ""),
                                     palette="tab20", with_overlap_curves=VIZ_OVERLAP_CURVES)
    route_plots.plot_plain_route_set(axes[0, 1], _ref_rr, _coords, _adj, title=_ref_key,
                                     subtitle=_sub.get(_ref_key, ""), palette="tab20",
                                     with_overlap_curves=VIZ_OVERLAP_CURVES)
    route_plots.plot_route_diff(axes[0, 2], _nb_rr, _ref_rr, _coords, _adj,
                                title="neural BCO -- diff vs init", subtitle=_sub.get(_nbco_key, ""),
                                palette="tab20", with_overlap_curves=VIZ_OVERLAP_CURVES)
    route_plots.plot_route_diff(axes[0, 3], _our_rr, _ref_rr, _coords, _adj,
                                title="Our NBCO -- diff vs init", subtitle=_sub.get(_our_key, ""),
                                palette="tab20", with_overlap_curves=VIZ_OVERLAP_CURVES)
    fig.suptitle(f"{VIZ_CITY} UNIFIED: routes / init / neural-BCO diff / our-NBCO diff",
                 fontsize=15, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show(); plt.close(fig)
    print(f"focused 4-panel: our={_our_key!r}, nbco={_nbco_key!r}, ref={_ref_key!r}")
else:
    print(f"4-panel skipped (missing key): our={_our_key!r}, nbco={_nbco_key!r}")

print(f"drawn demand + init + {len(_methods)} methods x2 (plain+diff) for {VIZ_CITY}; ref={_ref_key!r}")
